In [0]:
%python
# =============================================================================
# Setup: Create Query Parameter Widgets
# =============================================================================
# Purpose: Define notebook parameters for workspace and date range filtering
# =============================================================================

# Remove existing widgets (if any)
dbutils.widgets.removeAll()

# Create workspace_id_filter dropdown
dbutils.widgets.dropdown(
    name="workspace_id_filter",
    defaultValue="1516413757355523",
    choices=["1516413757355523","984752964297111"],  # Add more workspace IDs as needed
    label="Workspace ID"
)

# Create days_lookback dropdown
dbutils.widgets.dropdown(
    name="days_lookback",
    defaultValue="30",
    choices=["7", "30", "60", "90"],
    label="Days Lookback"
)

print("✅ Dashboard widgets created (for Categories 1–4):")
print(f"   workspace_id_filter = {dbutils.widgets.get('workspace_id_filter')}")
print(f"   days_lookback = {dbutils.widgets.get('days_lookback')}")
print(f"\n💡 Data extraction (Category 5) uses code-based config (see Config cell).")

✅ Query parameter widgets created:
   workspace_id_filter = 984752964297111
   days_lookback = 30

💡 Use the dropdown widgets above to change filters
   All queries in this notebook will use these parameters


# Genie Observability: System Tables & Architecture

## Overview

This notebook analyzes **aibiGenie** usage across three Databricks system tables to provide comprehensive observability for Genie spaces, conversations, and messages.

---

## System Tables Used

### 1. **`system.access.audit`** - Primary Data Source

**Purpose:** Records all actions and events occurring within Databricks workspaces, including every aibiGenie interaction.

**Key Columns:**
* `workspace_id` (STRING) - Workspace identifier
* `service_name` (STRING) - Filter: `'aibiGenie'` for Genie actions
* `action_name` (STRING) - Specific action performed (70+ Genie actions)
* `event_time` (TIMESTAMP) - When the action occurred
* `event_date` (DATE) - **Partition key** for efficient filtering
* `user_identity.email` (STRING) - User who performed the action
* `session_id` (STRING) - Groups related actions in a session
* `request_params` (MAP<STRING, STRING>) - Action parameters:
  * `request_params['space_id']` - Genie space identifier
  * `request_params['conversation_id']` - Conversation identifier
  * `request_params['message_id']` - Message identifier
* `response.status_code` (INT) - HTTP status (200=success, 400+=error)
* `response.error_message` (STRING) - Error details if failed

**Hierarchy Filtering:**
* **Space-level:** `space_id IS NOT NULL`, `conversation_id IS NULL`, `message_id IS NULL`
* **Conversation-level:** `space_id IS NOT NULL`, `conversation_id IS NOT NULL`, `message_id IS NULL`
* **Message-level:** `space_id IS NOT NULL`, `conversation_id IS NOT NULL`, `message_id IS NOT NULL`

**70+ Genie Actions Tracked:**
* Space: `createSpace`, `updateSpace`, `deleteSpace`, `getSpace`, `cloneSpace`, etc.
* Conversation: `createConversation`, `updateConversation`, `deleteConversation`, etc.
* Message: `createConversationMessage`, `executeMessageQuery`, `updateConversationMessageFeedback`, etc.
* Query: `executeMessageQuery`, `getMessageQueryResult`, `summarizeSqlExecutionResults`, etc.
* Metadata: `createInstruction`, `createCuratedQuestion`, `listInstructions`, etc.

---

### 2. **`system.query.history`** - SQL Query Execution Data

**Purpose:** Records every SQL query executed in SQL warehouses, including queries generated by Genie.

**Key Columns:**
* `workspace_id` (STRING) - Workspace identifier
* `statement_id` (STRING) - Unique query execution ID
* `executed_by` (STRING) - User email who ran the query
* `session_id` (STRING) - Spark session ID
* `start_time` (TIMESTAMP) - Query start time
* `end_time` (TIMESTAMP) - Query completion time
* `execution_status` (STRING) - FINISHED, FAILED, or CANCELED
* `compute.warehouse_id` (STRING) - SQL warehouse used
* `total_duration_ms` (LONG) - Query execution time in milliseconds
* `read_rows` (LONG) - Number of rows read
* `written_rows` (LONG) - Number of rows written
* `read_bytes` (LONG) - Data volume read
* `error_message` (STRING) - Error details if failed

**Use Case:** Link Genie message actions to actual SQL queries executed to analyze query performance and success rates.

---

### 3. **`system.access.workspaces_latest`** - Workspace Metadata

**Purpose:** Slow-changing dimension table containing metadata for all workspaces in the account.

**Key Columns:**
* `workspace_id` (STRING) - Workspace identifier
* `workspace_name` (STRING) - **Human-readable workspace name**
* `workspace_url` (STRING) - Workspace URL
* `status` (STRING) - Lifecycle status (RUNNING, PROVISIONING, etc.)
* `create_time` (TIMESTAMP) - When workspace was created

**Use Case:** Enrich audit logs with human-readable workspace names for reporting.

---

## How Tables Connect

### Primary Join Keys:

```
system.access.audit (ae)
    ├─ workspace_id ──────────┐
    ├─ user_identity.email ───┤
    ├─ session_id ────────────┤
    └─ event_time ────────────┤
                              │
                              ├──> system.query.history (qh)
                              │        ├─ workspace_id
                              │        ├─ executed_by
                              │        ├─ session_id
                              │        └─ start_time
                              │
                              └──> system.access.workspaces_latest (w)
                                       └─ workspace_id
```

### Join Strategy:

1. **Workspace Enrichment (Simple Join):**
   ```sql
   LEFT JOIN system.access.workspaces_latest w
     ON ae.workspace_id = w.workspace_id
   ```

2. **Query Correlation (Complex Join):**
   ```sql
   LEFT JOIN system.query.history qh
     ON ae.workspace_id = qh.workspace_id
     AND ae.user_identity.email = qh.executed_by
     AND ae.session_id = qh.session_id  -- Same session
     AND ABS(TIMESTAMPDIFF(SECOND, ae.event_time, qh.start_time)) <= 60  -- Within 60 seconds
   ```

**Why Temporal Correlation?** Genie actions (like `executeMessageQuery`) trigger SQL queries. By joining on session + time proximity, we link user actions to their resulting queries.

---

## Key Values & Filters

### Critical Filters for Performance:

```sql
WHERE ae.service_name = 'aibiGenie'  -- Only Genie actions
  AND ae.workspace_id = :workspace_id_filter  -- Specific workspace
  AND ae.event_date >= date_sub(CURRENT_DATE(), :days_lookback)  -- Partition pruning
  AND ae.event_date <= CURRENT_DATE()
```

**⚠️ Always filter on `event_date`** - it's the partition key for `system.access.audit`.

### Hierarchy Identification:

```sql
CASE 
  WHEN request_params['message_id'] IS NOT NULL THEN 'message'
  WHEN request_params['conversation_id'] IS NOT NULL THEN 'conversation'
  WHEN request_params['space_id'] IS NOT NULL THEN 'space'
  ELSE 'other'
END as hierarchy_level
```

---

## How This Supports Strategic Objectives

### 🏗️ **Option 1: Space CRUD Operations (Governance)**
* **Data Source:** `system.access.audit` filtered for space-level actions
* **Key Actions:** `createSpace`, `updateSpace`, `deleteSpace`, `cloneSpace`
* **Metrics:** Spaces created, deleted, modified, accessed
* **Value:** Track space sprawl, audit configuration changes, monitor adoption

### 📊 **Option 2: Active Spaces (Engagement)**
* **Data Source:** `system.access.audit` for conversation + message actions
* **Key Actions:** `createConversation`, `createConversationMessage`, `executeMessageQuery`
* **Metrics:** Conversations per space, messages per conversation, query execution rate
* **Value:** Identify active vs. dead spaces, measure user engagement, calculate ROI

### 🚀 **Option 3: Overall Genie Adoption (Platform Success)**
* **Data Source:** All three tables joined
* **Key Metrics:** 
  * Daily Active Users (DAU) - `COUNT(DISTINCT user_email)`
  * Query volume - from `system.query.history`
  * Error rates - `status_code >= 400`
  * Query performance - `total_duration_ms`, `read_rows`
* **Value:** Demonstrate platform value, track adoption trends, identify performance issues

---

## Data Limitations

### ⚠️ **Missing: Human-Readable Space Names**
* `system.access.audit` only contains `space_id` (UUID)
* **Solution:** Build a lookup table using Genie REST API:
  ```python
  # Fetch space metadata via API
  GET /api/2.0/genie/spaces/{space_id}
  # Store: space_id, space_name, workspace_id
  ```

### ⚠️ **Message Content Not Available**
* Audit logs don't contain actual message text or query SQL
* Only action metadata (who, when, what action, success/failure)
* **Solution:** Use Genie REST API for detailed message content if needed

---

## Query Performance Tips

1. **Always use partition filters:** `event_date >= date_sub(CURRENT_DATE(), 30)`
2. **Filter on workspace_id early:** Reduces data scanned
3. **Use CTEs for readability:** Break complex queries into logical steps
4. **Limit result sets:** Use `LIMIT` for exploratory queries
5. **Consider materialized views:** For frequently-run aggregations

---

**Next:** See Cell 2 for strategic objectives and query categories.

# Genie Observability Dashboard - Foundational Queries

## Strategic Objectives

This notebook addresses **three critical analytics needs**:

### 🏗️ **Option 1: Space CRUD Operations (Governance)**
* **Category 1** tracks space lifecycle events
* **Why:** Understand space creation trends, configuration changes, deletion patterns
* **Value:** Governance, space sprawl management, adoption tracking

### 📊 **Option 2: Active Spaces (Engagement)**
* **Category 2 & 3** track conversations and messages
* **Category 4** joins to identify spaces with actual activity
* **Why:** Distinguish between created spaces vs actively used spaces
* **Value:** ROI metrics, resource optimization, identify dead spaces

### 🚀 **Option 3: Overall Genie Adoption (Platform Success)**
* **Category 4** provides cross-hierarchy analytics
* **Why:** Measure platform adoption, user engagement, usage patterns
* **Value:** Demonstrate business value, track DAU/MAU, query volume trends

---

## 4 Query Categories

### 1. **Space-Level Actions** (Category 1)
   * Tracks: Space creation, updates, deletion, cloning
   * Excludes: Read-only operations (getSpace, listInstructions)
   * **Key Metrics:** Spaces created, spaces deleted, net space growth

### 2. **Conversation-Level Actions** (Category 2)
   * Tracks: Conversation creation, updates, deletion
   * **Key Metrics:** Conversations per space, conversation lifecycle

### 3. **Message-Level Actions** (Category 3)
   * Tracks: Message creation, queries, feedback, regeneration
   * **Key Metrics:** Message volume, query execution, feedback rates

### 4. **Cross-Hierarchy Analytics** (Category 4)
   * Joins: All levels + query history + workspace metadata
   * **Key Metrics:** DAU, active spaces, query performance, error rates

---

## Parameters
* **workspace_id_filter**: Filter by specific workspace (dropdown)
* **days_lookback**: 30, 60, or 90 days (dropdown)

---

In [0]:
-- Query parameters for dashboard filtering
-- Use the dropdown widgets above to change workspace and date range
-- Parameters: :workspace_id_filter and :days_lookback

-- Verify current parameter values
SELECT 
  :workspace_id_filter as workspace_id_filter,
  :days_lookback as days_lookback,
  date_sub(CURRENT_DATE(), :days_lookback) as start_date,
  CURRENT_DATE() as end_date

workspace_id_filter,days_lookback,start_date,end_date
1516413757355523,30,2026-01-01,2026-01-31


## Category 1: Space-Level Actions (Space CRUD)

Tracks **space lifecycle events** - creation, updates, deletion, cloning.

**Filter Criteria:**
* `service_name = 'aibiGenie'`
* `space_id IS NOT NULL`
* `conversation_id IS NULL`
* `message_id IS NULL`
* **Focus on lifecycle actions only**

**Tracked Actions (from official list):**
* **Create:** `createSpace`, `genieCreateSpace`
* **Update:** `updateSpace`, `genieUpdateSpace`, `updateGenieColumnConfigs`, `updateSampleQuestions`
* **Delete:** `deleteSpace`, `trashSpace`, `genieTrashSpace`
* **Clone:** `cloneSpace`
* **Retrieve:** `getSpace`, `genieGetSpace`, `genieListSpaces`

**Excluded Actions:**
* `listInstructions`, `listCuratedQuestions` - Metadata browsing (tracked in message-level)
* `listGenieSpaceMessages` - Message operations (tracked in message-level)

**Value:** Understand space creation trends, configuration changes, deletion patterns for governance.

In [0]:
-- Extract all space-level actions (lifecycle + read operations)
-- Based on official Genie action list

SELECT 
  -- Identifiers
  ae.workspace_id,
  CAST(ae.request_params['space_id'] AS STRING) as space_id,
  
  -- Temporal
  ae.event_date,
  ae.event_time,
  DATE_TRUNC('hour', ae.event_time) as event_hour,
  
  -- Action details
  ae.action_name,
  ae.request_id,
  
  -- User context
  ae.user_identity.email as user_email,
  ae.session_id,
  
  -- Response tracking
  ae.response.status_code as status_code,
  ae.response.error_message as error_message,
  
  -- Classification based on official action list
  CASE 
    WHEN ae.action_name IN ('createSpace', 'genieCreateSpace') THEN 'create'
    WHEN ae.action_name IN ('updateSpace', 'genieUpdateSpace', 'updateGenieColumnConfigs', 'updateSampleQuestions') THEN 'update'
    WHEN ae.action_name IN ('deleteSpace', 'trashSpace', 'genieTrashSpace') THEN 'delete'
    WHEN ae.action_name = 'cloneSpace' THEN 'clone'
    WHEN ae.action_name IN ('getSpace', 'genieGetSpace', 'genieListSpaces') THEN 'read'
    ELSE 'other'
  END as action_category
  
FROM system.access.audit ae

WHERE ae.service_name = 'aibiGenie'
  AND ae.request_params['space_id'] IS NOT NULL
  AND ae.request_params['conversation_id'] IS NULL
  AND ae.request_params['message_id'] IS NULL
  -- All space-level actions from official list
  AND ae.action_name IN (
    -- Create
    'createSpace', 'genieCreateSpace',
    -- Update
    'updateSpace', 'genieUpdateSpace', 'updateGenieColumnConfigs', 'updateSampleQuestions',
    -- Delete
    'deleteSpace', 'trashSpace', 'genieTrashSpace',
    -- Clone
    'cloneSpace',
    -- Read
    'getSpace', 'genieGetSpace', 'genieListSpaces'
  )
  AND ae.workspace_id = :workspace_id_filter
  AND ae.event_date >= date_sub(CURRENT_DATE(), :days_lookback)
  AND ae.event_date <= CURRENT_DATE()
  
ORDER BY ae.event_time DESC
LIMIT 1000

workspace_id,space_id,event_date,event_time,event_hour,action_name,request_id,user_email,session_id,status_code,error_message,action_category
1516413757355523,01f0aaabfed41a778b2e5302795ce495,2026-01-29,2026-01-29T18:57:07.575Z,2026-01-29T18:00:00.000Z,getSpace,cb27af4d-dd8a-4c08-a595-ba37bbb3f551,bhavin.kukadia@databricks.com,ephemeral-70d8f73a-1544-430c-9992-c5bf75e2752f,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,2026-01-29,2026-01-29T18:57:06.879Z,2026-01-29T18:00:00.000Z,getSpace,0111422d-2355-453d-9021-e84e29f121b3,bhavin.kukadia@databricks.com,ephemeral-5c8e1ded-4b23-4d69-8350-ec194671051f,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,2026-01-29,2026-01-29T18:57:06.716Z,2026-01-29T18:00:00.000Z,getSpace,0c186ae1-a751-443d-a253-f29ebc6e3d5d,bhavin.kukadia@databricks.com,ephemeral-381f0aa8-a7de-4f53-a411-05b047fc5c15,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,2026-01-29,2026-01-29T18:38:24.205Z,2026-01-29T18:00:00.000Z,getSpace,d21cdd3f-91c5-435b-8ade-d595ad6c0187,bhavin.kukadia@databricks.com,ephemeral-f1f4fe7e-fbf0-4220-be79-0461c1a70edb,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,2026-01-29,2026-01-29T17:18:16.884Z,2026-01-29T17:00:00.000Z,getSpace,e228f79b-2072-41b5-8c6a-3d78c376e0ac,bhavin.kukadia@databricks.com,ephemeral-485d3172-f8f5-4c97-a34f-adbfd13677d4,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,2026-01-29,2026-01-29T17:18:16.663Z,2026-01-29T17:00:00.000Z,getSpace,1b92fa9e-a2a9-4f3d-b2d0-449dd656fc79,bhavin.kukadia@databricks.com,ephemeral-948f22d7-c955-4455-9a24-d819c2abb31c,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,2026-01-29,2026-01-29T17:18:16.530Z,2026-01-29T17:00:00.000Z,getSpace,40ebe2d0-a8d2-47b5-8eb6-c5ce2919a6f9,bhavin.kukadia@databricks.com,ephemeral-dee3350d-ade9-453f-8537-fa3171569500,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,2026-01-29,2026-01-29T17:01:42.704Z,2026-01-29T17:00:00.000Z,getSpace,5adfbac1-44ba-4a51-be38-e32e1e2444ec,bhavin.kukadia@databricks.com,ephemeral-39cb14f9-c30c-448c-acfe-398f6bf20cda,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,2026-01-29,2026-01-29T17:00:53.401Z,2026-01-29T17:00:00.000Z,getSpace,d534aac1-8046-4bfc-8be3-95929f583572,bhavin.kukadia@databricks.com,ephemeral-4fa6b693-a1e0-45a5-b94d-7aff6ff96cdc,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,2026-01-29,2026-01-29T17:00:53.246Z,2026-01-29T17:00:00.000Z,getSpace,9b9482a6-91a5-4668-be07-0189c2c827fa,bhavin.kukadia@databricks.com,ephemeral-4d2832c0-ff9b-4aed-b9da-48a98fdb120c,200,null,read


In [0]:
-- Summary metrics for space-level actions
-- Key KPIs: spaces created/updated/deleted/accessed

WITH space_actions AS (
  SELECT 
    ae.workspace_id,
    CAST(ae.request_params['space_id'] AS STRING) as space_id,
    ae.event_date,
    ae.action_name,
    ae.user_identity.email as user_email,
    ae.response.status_code as status_code,
    CASE 
      WHEN ae.action_name IN ('createSpace', 'genieCreateSpace') THEN 'create'
      WHEN ae.action_name IN ('updateSpace', 'genieUpdateSpace', 'updateGenieColumnConfigs', 'updateSampleQuestions') THEN 'update'
      WHEN ae.action_name IN ('deleteSpace', 'trashSpace', 'genieTrashSpace') THEN 'delete'
      WHEN ae.action_name = 'cloneSpace' THEN 'clone'
      WHEN ae.action_name IN ('getSpace', 'genieGetSpace', 'genieListSpaces') THEN 'read'
      ELSE 'other'
    END as action_category
  FROM system.access.audit ae
  WHERE ae.service_name = 'aibiGenie'
    AND ae.request_params['space_id'] IS NOT NULL
    AND ae.request_params['conversation_id'] IS NULL
    AND ae.request_params['message_id'] IS NULL
    AND ae.action_name IN (
      'createSpace', 'genieCreateSpace',
      'updateSpace', 'genieUpdateSpace', 'updateGenieColumnConfigs', 'updateSampleQuestions',
      'deleteSpace', 'trashSpace', 'genieTrashSpace',
      'cloneSpace',
      'getSpace', 'genieGetSpace', 'genieListSpaces'
    )
    AND ae.workspace_id = :workspace_id_filter
    AND ae.event_date >= date_sub(CURRENT_DATE(), :days_lookback)
    AND ae.event_date <= CURRENT_DATE()
)

SELECT 
  -- Workspace context
  workspace_id,
  
  -- Space metrics
  COUNT(DISTINCT space_id) as unique_spaces,
  COUNT(DISTINCT CASE WHEN action_category = 'create' THEN space_id END) as spaces_created,
  COUNT(DISTINCT CASE WHEN action_category = 'delete' THEN space_id END) as spaces_deleted,
  COUNT(DISTINCT CASE WHEN action_category IN ('create', 'update', 'delete', 'clone') THEN space_id END) as spaces_modified,
  COUNT(DISTINCT CASE WHEN action_category = 'read' THEN space_id END) as spaces_accessed,
  
  -- Action distribution
  COUNT(*) as total_actions,
  COUNT(CASE WHEN action_category = 'create' THEN 1 END) as create_actions,
  COUNT(CASE WHEN action_category = 'update' THEN 1 END) as update_actions,
  COUNT(CASE WHEN action_category = 'delete' THEN 1 END) as delete_actions,
  COUNT(CASE WHEN action_category = 'clone' THEN 1 END) as clone_actions,
  COUNT(CASE WHEN action_category = 'read' THEN 1 END) as read_actions,
  
  -- User engagement
  COUNT(DISTINCT user_email) as unique_users,
  
  -- Error tracking
  COUNT(CASE WHEN status_code >= 400 THEN 1 END) as error_count,
  ROUND(COUNT(CASE WHEN status_code >= 400 THEN 1 END) * 100.0 / COUNT(*), 2) as error_rate_pct,
  
  -- Time range
  MIN(event_date) as first_event_date,
  MAX(event_date) as last_event_date
  
FROM space_actions
GROUP BY workspace_id

workspace_id,unique_spaces,spaces_created,spaces_deleted,spaces_modified,spaces_accessed,total_actions,create_actions,update_actions,delete_actions,clone_actions,read_actions,unique_users,error_count,error_rate_pct,first_event_date,last_event_date
1516413757355523,3,0,0,1,2,57,0,1,0,0,56,2,1,1.75,2026-01-08,2026-01-29


## Category 2: Conversation-Level Actions (Option 2: Engagement)

Tracks actions at the **conversation level** (within a space, but no message context).

**Filter Criteria:**
* `service_name = 'aibiGenie'`
* `space_id IS NOT NULL`
* `conversation_id IS NOT NULL`
* `message_id IS NULL`

**Tracked Actions (from official list):**
* **Create:** `createConversation`
* **Update:** `updateConversation`
* **Delete:** `deleteConversation`
* **Read:** `getConversation`, `listConversations`, `genieListConversations`

**Value:** Track conversation creation trends, identify active spaces with conversations.

In [0]:
-- Extract all conversation-level actions
-- Use this as the base for conversation-level analytics

SELECT 
  -- Identifiers
  ae.workspace_id,
  CAST(ae.request_params['space_id'] AS STRING) as space_id,
  CAST(ae.request_params['conversation_id'] AS STRING) as conversation_id,
  
  -- Temporal
  ae.event_date,
  ae.event_time,
  DATE_TRUNC('hour', ae.event_time) as event_hour,
  
  -- Action details
  ae.action_name,
  ae.request_id,
  
  -- User context
  ae.user_identity.email as user_email,
  ae.session_id,
  
  -- Response tracking
  ae.response.status_code as status_code,
  ae.response.error_message as error_message,
  
  -- Classification
  CASE 
    WHEN ae.action_name = 'createConversation' THEN 'create'
    WHEN ae.action_name = 'updateConversation' THEN 'update'
    WHEN ae.action_name = 'deleteConversation' THEN 'delete'
    WHEN ae.action_name IN ('getConversation', 'listConversations', 'genieListConversations') THEN 'read'
    ELSE 'other'
  END as action_category
  
FROM system.access.audit ae

WHERE ae.service_name = 'aibiGenie'
  AND ae.request_params['space_id'] IS NOT NULL
  AND ae.request_params['conversation_id'] IS NOT NULL
  AND ae.request_params['message_id'] IS NULL
  AND ae.workspace_id = :workspace_id_filter
  AND ae.event_date >= date_sub(CURRENT_DATE(), :days_lookback)
  AND ae.event_date <= CURRENT_DATE()
  
ORDER BY ae.event_time DESC
LIMIT 1000

workspace_id,space_id,conversation_id,event_date,event_time,event_hour,action_name,request_id,user_email,session_id,status_code,error_message,action_category
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,2026-01-29,2026-01-29T17:03:13.205Z,2026-01-29T17:00:00.000Z,createConversationMessage,5032ed21-8724-4e66-ac1a-cf0ed3378c5e,bhavin.kukadia@databricks.com,ephemeral-beabb5be-bc4f-444a-be12-4f3206335d84,200,null,other
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,2026-01-29,2026-01-29T17:02:57.127Z,2026-01-29T17:00:00.000Z,createConversationMessage,452ee857-dd04-4b19-9df5-ea7da3a2a82f,bhavin.kukadia@databricks.com,ephemeral-371dcf06-3e8a-4722-ab57-532779a3c427,200,null,other
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,2026-01-29,2026-01-29T17:02:28.678Z,2026-01-29T17:00:00.000Z,createConversationMessage,332f4e3c-94cb-42bb-9113-06ef3a555bbd,bhavin.kukadia@databricks.com,ephemeral-02c90f96-3f8a-4131-8be1-03f40d446450,200,null,other
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,2026-01-29,2026-01-29T17:02:17.835Z,2026-01-29T17:00:00.000Z,createConversationMessage,d3ec702e-01d5-4173-96e0-17c780926b7f,bhavin.kukadia@databricks.com,ephemeral-3bd2e328-c65b-4b80-8db0-3b1cbfd57fbc,200,null,other
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,2026-01-29,2026-01-29T17:01:45.584Z,2026-01-29T17:00:00.000Z,createConversationMessage,ebdb5536-04c7-43c3-ad23-d394ba7a0843,bhavin.kukadia@databricks.com,ephemeral-548742dc-8c32-49cb-a2a4-4d1979bc80c6,200,null,other
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,2026-01-29,2026-01-29T17:01:02.775Z,2026-01-29T17:00:00.000Z,createConversationMessage,f0b00f15-46be-44b9-86aa-70d16de6fc07,bhavin.kukadia@databricks.com,ephemeral-af9eeead-c647-4774-81ee-aca581b5c512,200,null,other
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,2026-01-29,2026-01-29T17:01:02.705Z,2026-01-29T17:00:00.000Z,getConversation,cb07772d-80d5-41ea-8492-5f3eb7afeab5,bhavin.kukadia@databricks.com,ephemeral-b2e4e429-c7f6-4274-a533-5418291abc62,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fb2a898d1003ac51b7d175175879,2026-01-29,2026-01-29T02:12:32.630Z,2026-01-29T02:00:00.000Z,getConversation,7e7e7c7a-ce9e-48fa-a522-3a9cc4730066,bhavin.kukadia@databricks.com,ephemeral-2068f2eb-21e6-4147-b1a9-0c8f220823e4,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fb2a898d1003ac51b7d175175879,2026-01-29,2026-01-29T01:58:09.846Z,2026-01-29T01:00:00.000Z,getConversation,92c41313-5e09-4d5e-bac1-f30bf04fc731,bhavin.kukadia@databricks.com,ephemeral-c38ab765-7773-45a4-814c-ed867afcf1b6,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fb2a898d1003ac51b7d175175879,2026-01-29,2026-01-29T00:59:57.625Z,2026-01-29T00:00:00.000Z,getConversation,53987e20-3c16-4289-845d-edd5345e0435,bhavin.kukadia@databricks.com,ephemeral-c5aca7b0-35f7-49c1-891a-83028704bfd7,200,null,read


In [0]:
-- Summary metrics for conversation-level actions
-- Key KPIs: conversation count per space, creation trends, engagement

WITH conversation_actions AS (
  SELECT 
    ae.workspace_id,
    CAST(ae.request_params['space_id'] AS STRING) as space_id,
    CAST(ae.request_params['conversation_id'] AS STRING) as conversation_id,
    ae.event_date,
    ae.action_name,
    ae.user_identity.email as user_email,
    ae.response.status_code as status_code,
    CASE 
      WHEN ae.action_name = 'createConversation' THEN 'create'
      WHEN ae.action_name = 'updateConversation' THEN 'update'
      WHEN ae.action_name = 'deleteConversation' THEN 'delete'
      WHEN ae.action_name IN ('getConversation', 'listConversations', 'genieListConversations') THEN 'read'
      ELSE 'other'
    END as action_category
  FROM system.access.audit ae
  WHERE ae.service_name = 'aibiGenie'
    AND ae.request_params['space_id'] IS NOT NULL
    AND ae.request_params['conversation_id'] IS NOT NULL
    AND ae.request_params['message_id'] IS NULL
    AND ae.workspace_id = :workspace_id_filter
    AND ae.event_date >= date_sub(CURRENT_DATE(), :days_lookback)
    AND ae.event_date <= CURRENT_DATE()
)

SELECT 
  -- Workspace context
  workspace_id,
  
  -- Space metrics
  COUNT(DISTINCT space_id) as spaces_with_conversations,
  
  -- Conversation metrics
  COUNT(DISTINCT conversation_id) as unique_conversations,
  COUNT(DISTINCT CASE WHEN action_category = 'create' THEN conversation_id END) as conversations_created,
  ROUND(COUNT(DISTINCT conversation_id) * 1.0 / NULLIF(COUNT(DISTINCT space_id), 0), 2) as avg_conversations_per_space,
  
  -- Action distribution
  COUNT(*) as total_actions,
  COUNT(CASE WHEN action_category = 'create' THEN 1 END) as create_actions,
  COUNT(CASE WHEN action_category = 'update' THEN 1 END) as update_actions,
  COUNT(CASE WHEN action_category = 'delete' THEN 1 END) as delete_actions,
  COUNT(CASE WHEN action_category = 'read' THEN 1 END) as read_actions,
  
  -- User engagement
  COUNT(DISTINCT user_email) as unique_users,
  
  -- Error tracking
  COUNT(CASE WHEN status_code >= 400 THEN 1 END) as error_count,
  ROUND(COUNT(CASE WHEN status_code >= 400 THEN 1 END) * 100.0 / COUNT(*), 2) as error_rate_pct,
  
  -- Time range
  MIN(event_date) as first_event_date,
  MAX(event_date) as last_event_date
  
FROM conversation_actions
GROUP BY workspace_id

workspace_id,spaces_with_conversations,unique_conversations,conversations_created,avg_conversations_per_space,total_actions,create_actions,update_actions,delete_actions,read_actions,unique_users,error_count,error_rate_pct,first_event_date,last_event_date
1516413757355523,2,7,0,3.50,49,0,0,0,15,2,0,0.00,2026-01-23,2026-01-29


## Category 3: Message-Level Actions (Engagement)

Tracks actions at the **message level** (most granular - within conversations).

**Filter Criteria:**
* `service_name = 'aibiGenie'`
* `space_id IS NOT NULL`
* `conversation_id IS NOT NULL`
* `message_id IS NOT NULL`

**Tracked Actions (from official list):**
* **Create:** `createConversationMessage`, `genieCreateConversationMessage`, `genieStartConversationMessage`
* **Update:** `updateConversationMessage`, `regenerateConversationMessage`, `updateMessageAttachment`
* **Delete:** `deleteConversationMessage`
* **Cancel:** `cancelMessage`
* **Read:** `getConversationMessage`, `genieGetConversationMessage`, `genieListConversationMessages`, `listGenieSpaceMessages`, `listGenieSpaceUserMessages`
* **Query Execution:** `executeMessageQuery`, `executeMessageAttachmentQuery`, `genieExecuteMessageQuery`, `genieExecuteMessageAttachmentQuery`, `executeQuery`, `executeFullQueryResult`
* **Query Results:** `getMessageQueryResult`, `getMessageAttachmentQueryResult`, `genieGetMessageQueryResult`, `genieGetMessageAttachmentQueryResult`, `genieGetQueryResultByAttachment`, `getQueryResult`, `genieGenerateDownloadFullQueryResult`, `summarizeSqlExecutionResults`
* **Feedback:** `updateConversationMessageFeedback`, `genieSendMessageFeedback`
* **Comments:** `createConversationMessageComment`, `deleteConversationMessageComment`, `listConversationMessageComments`
* **Instructions/Questions:** `createInstruction`, `updateInstruction`, `deleteInstruction`, `listInstructions`, `createCuratedQuestion`, `updateCuratedQuestion`, `listCuratedQuestions`

**Value:** Track user engagement, query execution patterns, feedback rates, instruction usage.

In [0]:
-- Extract all message-level actions
-- Based on official Genie action list - comprehensive message tracking

SELECT 
  -- Identifiers
  ae.workspace_id,
  CAST(ae.request_params['space_id'] AS STRING) as space_id,
  CAST(ae.request_params['conversation_id'] AS STRING) as conversation_id,
  CAST(ae.request_params['message_id'] AS STRING) as message_id,
  CAST(ae.request_params['attachment_id'] AS STRING) as attachment_id,
  
  -- Temporal
  ae.event_date,
  ae.event_time,
  DATE_TRUNC('hour', ae.event_time) as event_hour,
  
  -- Action details
  ae.action_name,
  ae.request_id,
  
  -- User context
  ae.user_identity.email as user_email,
  ae.session_id,
  
  -- Response tracking
  ae.response.status_code as status_code,
  ae.response.error_message as error_message,
  
  -- Comprehensive classification based on official action list
  CASE 
    -- Message CRUD
    WHEN ae.action_name IN ('createConversationMessage', 'genieCreateConversationMessage', 'genieStartConversationMessage') THEN 'create'
    WHEN ae.action_name IN ('updateConversationMessage', 'regenerateConversationMessage', 'updateMessageAttachment') THEN 'update'
    WHEN ae.action_name = 'deleteConversationMessage' THEN 'delete'
    WHEN ae.action_name = 'cancelMessage' THEN 'cancel'
    
    -- Message Read
    WHEN ae.action_name IN ('getConversationMessage', 'genieGetConversationMessage', 'genieListConversationMessages', 'listGenieSpaceMessages', 'listGenieSpaceUserMessages') THEN 'read'
    
    -- Query Execution
    WHEN ae.action_name IN ('executeMessageQuery', 'executeMessageAttachmentQuery', 'genieExecuteMessageQuery', 'genieExecuteMessageAttachmentQuery', 'executeQuery', 'executeFullQueryResult') THEN 'query_execute'
    
    -- Query Results
    WHEN ae.action_name IN ('getMessageQueryResult', 'getMessageAttachmentQueryResult', 'genieGetMessageQueryResult', 'genieGetMessageAttachmentQueryResult', 'genieGetQueryResultByAttachment', 'getQueryResult', 'genieGenerateDownloadFullQueryResult', 'summarizeSqlExecutionResults') THEN 'query_result'
    
    -- Feedback
    WHEN ae.action_name IN ('updateConversationMessageFeedback', 'genieSendMessageFeedback') THEN 'feedback'
    
    -- Comments
    WHEN ae.action_name IN ('createConversationMessageComment', 'deleteConversationMessageComment', 'listConversationMessageComments') THEN 'comment'
    
    -- Instructions & Curated Questions
    WHEN ae.action_name IN ('createInstruction', 'updateInstruction', 'deleteInstruction', 'listInstructions') THEN 'instruction'
    WHEN ae.action_name IN ('createCuratedQuestion', 'updateCuratedQuestion', 'listCuratedQuestions') THEN 'curated_question'
    
    ELSE 'other'
  END as action_category
  
FROM system.access.audit ae

WHERE ae.service_name = 'aibiGenie'
  AND ae.request_params['space_id'] IS NOT NULL
  AND (
    -- Message-level actions (has message_id)
    (ae.request_params['message_id'] IS NOT NULL AND CAST(ae.request_params['message_id'] AS STRING) != '')
    OR
    -- Space-level metadata actions that belong to message analytics
    ae.action_name IN ('listGenieSpaceMessages', 'listGenieSpaceUserMessages', 'listInstructions', 'listCuratedQuestions', 'createInstruction', 'updateInstruction', 'deleteInstruction', 'createCuratedQuestion', 'updateCuratedQuestion')
  )
  -- Filter out rows where conversation_id or message_id is null/empty
  AND (ae.request_params['conversation_id'] IS NOT NULL AND CAST(ae.request_params['conversation_id'] AS STRING) != '')
  AND (ae.request_params['message_id'] IS NOT NULL AND CAST(ae.request_params['message_id'] AS STRING) != '')
  AND ae.workspace_id = :workspace_id_filter
  AND ae.event_date >= date_sub(CURRENT_DATE(), :days_lookback)
  AND ae.event_date <= CURRENT_DATE()
  
ORDER BY ae.event_time DESC
LIMIT 1000

workspace_id,space_id,conversation_id,message_id,attachment_id,event_date,event_time,event_hour,action_name,request_id,user_email,session_id,status_code,error_message,action_category
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd3444851cf3be2a1048e5287172,null,2026-01-30,2026-01-30T19:43:39.569Z,2026-01-30T19:00:00.000Z,genieGetConversationMessage,aaa011cd-87b8-4bbe-a4e7-fb56df252826,40ed6418-a18c-403d-8137-254f4489df49,null,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd344afc14158579397bd3cacb76,null,2026-01-30,2026-01-30T19:43:39.271Z,2026-01-30T19:00:00.000Z,genieGetConversationMessage,f63daeea-63f1-4ff6-9808-4c735ad0eb1f,40ed6418-a18c-403d-8137-254f4489df49,null,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd3465861986a917b3d94ac7d6f4,null,2026-01-30,2026-01-30T19:43:38.972Z,2026-01-30T19:00:00.000Z,genieGetConversationMessage,2852a2bd-3598-401c-9877-d3fc2ea5fad2,40ed6418-a18c-403d-8137-254f4489df49,null,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd345bec143f83c0f72c9a55583d,null,2026-01-30,2026-01-30T19:43:38.669Z,2026-01-30T19:00:00.000Z,genieGetConversationMessage,9a07bb73-a749-40c0-a68d-9b5f5afe77fb,40ed6418-a18c-403d-8137-254f4489df49,null,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd3417c81620b1f043a30a77ddae,null,2026-01-30,2026-01-30T19:43:38.365Z,2026-01-30T19:00:00.000Z,genieGetConversationMessage,ba330802-1633-4a20-ba87-577b1f8818aa,40ed6418-a18c-403d-8137-254f4489df49,null,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd34314c1865834781b40658ae24,null,2026-01-30,2026-01-30T19:43:38.062Z,2026-01-30T19:00:00.000Z,genieGetConversationMessage,10383388-797e-49a6-b16e-7d1e116159b9,40ed6418-a18c-403d-8137-254f4489df49,null,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd3444851cf3be2a1048e5287172,null,2026-01-30,2026-01-30T19:43:37.757Z,2026-01-30T19:00:00.000Z,genieGetConversationMessage,f3174dd2-d2d2-4f15-9b3b-a50bfde764d8,40ed6418-a18c-403d-8137-254f4489df49,null,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd344afc14158579397bd3cacb76,null,2026-01-30,2026-01-30T19:43:37.448Z,2026-01-30T19:00:00.000Z,genieGetConversationMessage,7718195a-47c6-40fc-ac47-2b279cd723e5,40ed6418-a18c-403d-8137-254f4489df49,null,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd3417c81620b1f043a30a77ddae,null,2026-01-30,2026-01-30T19:43:37.151Z,2026-01-30T19:00:00.000Z,genieGetConversationMessage,4c72d1ad-4771-48b5-bbaf-d892b62f36be,40ed6418-a18c-403d-8137-254f4489df49,null,200,null,read
1516413757355523,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd3465861986a917b3d94ac7d6f4,null,2026-01-30,2026-01-30T19:43:36.841Z,2026-01-30T19:00:00.000Z,genieGetConversationMessage,6604d1e5-2f04-4bee-9f3c-9011e708cddd,40ed6418-a18c-403d-8137-254f4489df49,null,200,null,read


In [0]:
-- Summary metrics for message-level actions
-- Key KPIs: message volume, query execution, feedback rates, instruction usage

WITH message_actions AS (
  SELECT 
    ae.workspace_id,
    CAST(ae.request_params['space_id'] AS STRING) as space_id,
    CAST(ae.request_params['conversation_id'] AS STRING) as conversation_id,
    CAST(ae.request_params['message_id'] AS STRING) as message_id,
    ae.event_date,
    ae.action_name,
    ae.user_identity.email as user_email,
    ae.response.status_code as status_code,
    CASE 
      WHEN ae.action_name IN ('createConversationMessage', 'genieCreateConversationMessage', 'genieStartConversationMessage') THEN 'create'
      WHEN ae.action_name IN ('updateConversationMessage', 'regenerateConversationMessage', 'updateMessageAttachment') THEN 'update'
      WHEN ae.action_name = 'deleteConversationMessage' THEN 'delete'
      WHEN ae.action_name = 'cancelMessage' THEN 'cancel'
      WHEN ae.action_name IN ('getConversationMessage', 'genieGetConversationMessage', 'genieListConversationMessages', 'listGenieSpaceMessages', 'listGenieSpaceUserMessages') THEN 'read'
      WHEN ae.action_name IN ('executeMessageQuery', 'executeMessageAttachmentQuery', 'genieExecuteMessageQuery', 'genieExecuteMessageAttachmentQuery', 'executeQuery', 'executeFullQueryResult') THEN 'query_execute'
      WHEN ae.action_name IN ('getMessageQueryResult', 'getMessageAttachmentQueryResult', 'genieGetMessageQueryResult', 'genieGetMessageAttachmentQueryResult', 'genieGetQueryResultByAttachment', 'getQueryResult', 'genieGenerateDownloadFullQueryResult', 'summarizeSqlExecutionResults') THEN 'query_result'
      WHEN ae.action_name IN ('updateConversationMessageFeedback', 'genieSendMessageFeedback') THEN 'feedback'
      WHEN ae.action_name IN ('createConversationMessageComment', 'deleteConversationMessageComment', 'listConversationMessageComments') THEN 'comment'
      WHEN ae.action_name IN ('createInstruction', 'updateInstruction', 'deleteInstruction', 'listInstructions') THEN 'instruction'
      WHEN ae.action_name IN ('createCuratedQuestion', 'updateCuratedQuestion', 'listCuratedQuestions') THEN 'curated_question'
      ELSE 'other'
    END as action_category
  FROM system.access.audit ae
  WHERE ae.service_name = 'aibiGenie'
    AND ae.request_params['space_id'] IS NOT NULL
    AND (
      (ae.request_params['message_id'] IS NOT NULL AND CAST(ae.request_params['message_id'] AS STRING) != '')
      OR ae.action_name IN ('listGenieSpaceMessages', 'listGenieSpaceUserMessages', 'listInstructions', 'listCuratedQuestions', 'createInstruction', 'updateInstruction', 'deleteInstruction', 'createCuratedQuestion', 'updateCuratedQuestion')
    )
    AND (ae.request_params['conversation_id'] IS NOT NULL AND CAST(ae.request_params['conversation_id'] AS STRING) != '')
    AND (ae.request_params['message_id'] IS NOT NULL AND CAST(ae.request_params['message_id'] AS STRING) != '')
    AND ae.workspace_id = :workspace_id_filter
    AND ae.event_date >= date_sub(CURRENT_DATE(), :days_lookback)
    AND ae.event_date <= CURRENT_DATE()
)

SELECT 
  -- Workspace context
  workspace_id,
  
  -- Hierarchy metrics
  COUNT(DISTINCT space_id) as spaces_with_messages,
  COUNT(DISTINCT conversation_id) as conversations_with_messages,
  
  -- Message metrics
  COUNT(DISTINCT message_id) as unique_messages,
  COUNT(DISTINCT CASE WHEN action_category = 'create' THEN message_id END) as messages_created,
  ROUND(COUNT(DISTINCT message_id) * 1.0 / NULLIF(COUNT(DISTINCT conversation_id), 0), 2) as avg_messages_per_conversation,
  
  -- Action distribution
  COUNT(*) as total_actions,
  COUNT(CASE WHEN action_category = 'create' THEN 1 END) as create_actions,
  COUNT(CASE WHEN action_category = 'update' THEN 1 END) as update_actions,
  COUNT(CASE WHEN action_category = 'delete' THEN 1 END) as delete_actions,
  COUNT(CASE WHEN action_category = 'cancel' THEN 1 END) as cancel_actions,
  COUNT(CASE WHEN action_category = 'read' THEN 1 END) as read_actions,
  COUNT(CASE WHEN action_category = 'query_execute' THEN 1 END) as query_execute_actions,
  COUNT(CASE WHEN action_category = 'query_result' THEN 1 END) as query_result_actions,
  COUNT(CASE WHEN action_category = 'feedback' THEN 1 END) as feedback_actions,
  COUNT(CASE WHEN action_category = 'comment' THEN 1 END) as comment_actions,
  COUNT(CASE WHEN action_category = 'instruction' THEN 1 END) as instruction_actions,
  COUNT(CASE WHEN action_category = 'curated_question' THEN 1 END) as curated_question_actions,
  
  -- Engagement metrics
  COUNT(DISTINCT user_email) as unique_users,
  ROUND(COUNT(CASE WHEN action_category = 'feedback' THEN 1 END) * 100.0 / NULLIF(COUNT(CASE WHEN action_category = 'create' THEN 1 END), 0), 2) as feedback_rate_pct,
  ROUND(COUNT(CASE WHEN action_category = 'query_execute' THEN 1 END) * 100.0 / NULLIF(COUNT(CASE WHEN action_category = 'create' THEN 1 END), 0), 2) as query_execution_rate_pct,
  
  -- Error tracking
  COUNT(CASE WHEN status_code >= 400 THEN 1 END) as error_count,
  ROUND(COUNT(CASE WHEN status_code >= 400 THEN 1 END) * 100.0 / COUNT(*), 2) as error_rate_pct,
  
  -- Time range
  MIN(event_date) as first_event_date,
  MAX(event_date) as last_event_date
  
FROM message_actions
GROUP BY workspace_id

workspace_id,spaces_with_messages,conversations_with_messages,unique_messages,messages_created,avg_messages_per_conversation,total_actions,create_actions,update_actions,delete_actions,cancel_actions,read_actions,query_execute_actions,query_result_actions,feedback_actions,comment_actions,instruction_actions,curated_question_actions,unique_users,feedback_rate_pct,query_execution_rate_pct,error_count,error_rate_pct,first_event_date,last_event_date
1516413757355523,4,14,29,0,2.07,461,0,0,0,0,422,1,34,2,2,0,0,3,null,null,57,12.36,2026-01-23,2026-01-30


## Category 4: Cross-Hierarchy Analytics

Joins all Genie action levels with query history and workspace metadata for comprehensive observability.

**Key Joins:**
* `system.access.audit` (all Genie actions)
* `system.query.history` (SQL queries executed)
* `system.access.workspaces_latest` (workspace metadata)

**Join Keys:**
* `workspace_id` - Primary join key
* `session_id` - Links actions to queries in same session
* `user_identity.email` / `executed_by` - User correlation
* Temporal correlation (queries within 60 seconds of action)

**Use Cases:**
* End-to-end user journey tracking
* Query performance analysis per Genie action
* Error correlation across systems
* Workspace-level aggregations

In [0]:
-- Join all Genie actions with related query executions
-- Provides end-to-end observability from user action to query execution

WITH all_genie_actions AS (
  SELECT 
    ae.workspace_id,
    ae.event_time,
    ae.event_date,
    ae.user_identity.email as user_email,
    ae.session_id,
    ae.action_name,
    ae.request_id,
    
    -- Genie hierarchy
    CAST(ae.request_params['space_id'] AS STRING) as space_id,
    CAST(ae.request_params['conversation_id'] AS STRING) as conversation_id,
    CAST(ae.request_params['message_id'] AS STRING) as message_id,
    
    -- Response tracking
    ae.response.status_code as action_status_code,
    ae.response.error_message as action_error,
    
    -- Classify hierarchy level
    CASE 
      WHEN ae.request_params['message_id'] IS NOT NULL THEN 'message'
      WHEN ae.request_params['conversation_id'] IS NOT NULL THEN 'conversation'
      WHEN ae.request_params['space_id'] IS NOT NULL THEN 'space'
      ELSE 'other'
    END as hierarchy_level
    
  FROM system.access.audit ae
  WHERE ae.service_name = 'aibiGenie'
    AND ae.request_params['space_id'] IS NOT NULL
    AND ae.workspace_id = :workspace_id_filter
    AND ae.event_date >= date_sub(CURRENT_DATE(), :days_lookback)
    AND ae.event_date <= CURRENT_DATE()
),

related_queries AS (
  SELECT 
    qh.workspace_id,
    qh.statement_id,
    qh.executed_by,
    qh.session_id,
    qh.start_time,
    qh.end_time,
    qh.execution_status,
    qh.compute.warehouse_id,
    qh.total_duration_ms,
    qh.read_rows,
    qh.written_rows,
    qh.read_bytes,
    qh.error_message as query_error
  FROM system.query.history qh
  WHERE qh.workspace_id = :workspace_id_filter
    AND DATE(qh.start_time) >= date_sub(CURRENT_DATE(), :days_lookback)
    AND DATE(qh.start_time) <= CURRENT_DATE()
)

SELECT 
  -- Workspace context
  w.workspace_name,
  w.workspace_url,
  
  -- Genie hierarchy
  ga.space_id,
  ga.conversation_id,
  ga.message_id,
  ga.hierarchy_level,
  
  -- Action details
  ga.action_name,
  ga.event_time as action_time,
  ga.user_email,
  ga.action_status_code,
  ga.action_error,
  
  -- Related query details
  rq.statement_id,
  rq.execution_status as query_status,
  rq.total_duration_ms as query_duration_ms,
  rq.read_rows,
  rq.written_rows,
  rq.read_bytes,
  rq.query_error,
  
  -- Time correlation
  TIMESTAMPDIFF(SECOND, ga.event_time, rq.start_time) as seconds_between_action_and_query,
  
  -- Session correlation flag
  CASE WHEN ga.session_id = rq.session_id THEN 'same_session' ELSE 'different_session' END as session_match
  
FROM all_genie_actions ga

-- Join workspace metadata
LEFT JOIN system.access.workspaces_latest w
  ON ga.workspace_id = w.workspace_id

-- Join related queries (within 60 seconds and same user)
LEFT JOIN related_queries rq
  ON ga.workspace_id = rq.workspace_id
  AND ga.user_email = rq.executed_by
  AND ABS(TIMESTAMPDIFF(SECOND, ga.event_time, rq.start_time)) <= 60

ORDER BY ga.event_time DESC, rq.start_time DESC
LIMIT 1000

workspace_name,workspace_url,space_id,conversation_id,message_id,hierarchy_level,action_name,action_time,user_email,action_status_code,action_error,statement_id,query_status,query_duration_ms,read_rows,written_rows,read_bytes,query_error,seconds_between_action_and_query,session_match
DBW-BK-wx1,https://adb-1516413757355523.3.azuredatabricks.net,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd3444851cf3be2a1048e5287172,message,genieGetConversationMessage,2026-01-30T19:43:39.569Z,40ed6418-a18c-403d-8137-254f4489df49,200,null,null,null,null,null,null,null,null,null,different_session
DBW-BK-wx1,https://adb-1516413757355523.3.azuredatabricks.net,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd344afc14158579397bd3cacb76,message,genieGetConversationMessage,2026-01-30T19:43:39.271Z,40ed6418-a18c-403d-8137-254f4489df49,200,null,null,null,null,null,null,null,null,null,different_session
DBW-BK-wx1,https://adb-1516413757355523.3.azuredatabricks.net,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd3465861986a917b3d94ac7d6f4,message,genieGetConversationMessage,2026-01-30T19:43:38.972Z,40ed6418-a18c-403d-8137-254f4489df49,200,null,null,null,null,null,null,null,null,null,different_session
DBW-BK-wx1,https://adb-1516413757355523.3.azuredatabricks.net,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd345bec143f83c0f72c9a55583d,message,genieGetConversationMessage,2026-01-30T19:43:38.669Z,40ed6418-a18c-403d-8137-254f4489df49,200,null,null,null,null,null,null,null,null,null,different_session
DBW-BK-wx1,https://adb-1516413757355523.3.azuredatabricks.net,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd3417c81620b1f043a30a77ddae,message,genieGetConversationMessage,2026-01-30T19:43:38.365Z,40ed6418-a18c-403d-8137-254f4489df49,200,null,null,null,null,null,null,null,null,null,different_session
DBW-BK-wx1,https://adb-1516413757355523.3.azuredatabricks.net,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd34314c1865834781b40658ae24,message,genieGetConversationMessage,2026-01-30T19:43:38.062Z,40ed6418-a18c-403d-8137-254f4489df49,200,null,null,null,null,null,null,null,null,null,different_session
DBW-BK-wx1,https://adb-1516413757355523.3.azuredatabricks.net,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd3444851cf3be2a1048e5287172,message,genieGetConversationMessage,2026-01-30T19:43:37.757Z,40ed6418-a18c-403d-8137-254f4489df49,200,null,null,null,null,null,null,null,null,null,different_session
DBW-BK-wx1,https://adb-1516413757355523.3.azuredatabricks.net,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd344afc14158579397bd3cacb76,message,genieGetConversationMessage,2026-01-30T19:43:37.448Z,40ed6418-a18c-403d-8137-254f4489df49,200,null,null,null,null,null,null,null,null,null,different_session
DBW-BK-wx1,https://adb-1516413757355523.3.azuredatabricks.net,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd3417c81620b1f043a30a77ddae,message,genieGetConversationMessage,2026-01-30T19:43:37.151Z,40ed6418-a18c-403d-8137-254f4489df49,200,null,null,null,null,null,null,null,null,null,different_session
DBW-BK-wx1,https://adb-1516413757355523.3.azuredatabricks.net,01f0aaabfed41a778b2e5302795ce495,01f0fd34179e1edd8ded89bd6e85a671,01f0fd3465861986a917b3d94ac7d6f4,message,genieGetConversationMessage,2026-01-30T19:43:36.841Z,40ed6418-a18c-403d-8137-254f4489df49,200,null,null,null,null,null,null,null,null,null,different_session


In [0]:
-- Comprehensive summary across all hierarchy levels
-- Provides high-level KPIs for dashboard (Option 3: Overall Adoption)

WITH all_genie_actions AS (
  SELECT 
    ae.workspace_id,
    ae.event_date,
    ae.user_identity.email as user_email,
    ae.session_id,
    ae.action_name,
    CAST(ae.request_params['space_id'] AS STRING) as space_id,
    CAST(ae.request_params['conversation_id'] AS STRING) as conversation_id,
    CAST(ae.request_params['message_id'] AS STRING) as message_id,
    ae.response.status_code as status_code,
    CASE 
      WHEN ae.request_params['message_id'] IS NOT NULL AND CAST(ae.request_params['message_id'] AS STRING) != '' THEN 'message'
      WHEN ae.request_params['conversation_id'] IS NOT NULL AND CAST(ae.request_params['conversation_id'] AS STRING) != '' THEN 'conversation'
      WHEN ae.request_params['space_id'] IS NOT NULL THEN 'space'
      ELSE 'other'
    END as hierarchy_level
  FROM system.access.audit ae
  WHERE ae.service_name = 'aibiGenie'
    AND ae.request_params['space_id'] IS NOT NULL
    AND ae.workspace_id = :workspace_id_filter
    AND ae.event_date >= date_sub(CURRENT_DATE(), :days_lookback)
    AND ae.event_date <= CURRENT_DATE()
)

SELECT 
  workspace_id,
  
  -- Overall metrics (Option 3: Adoption)
  COUNT(*) as total_actions,
  COUNT(DISTINCT user_email) as unique_users,
  COUNT(DISTINCT session_id) as unique_sessions,
  
  -- Hierarchy breakdown (Option 1 & 2)
  COUNT(DISTINCT space_id) as unique_spaces,
  COUNT(DISTINCT conversation_id) as unique_conversations,
  COUNT(DISTINCT message_id) as unique_messages,
  
  -- Action distribution by level
  COUNT(CASE WHEN hierarchy_level = 'space' THEN 1 END) as space_level_actions,
  COUNT(CASE WHEN hierarchy_level = 'conversation' THEN 1 END) as conversation_level_actions,
  COUNT(CASE WHEN hierarchy_level = 'message' THEN 1 END) as message_level_actions,
  
  -- Engagement ratios (Option 2: Active Spaces)
  ROUND(COUNT(DISTINCT conversation_id) * 1.0 / NULLIF(COUNT(DISTINCT space_id), 0), 2) as conversations_per_space,
  ROUND(COUNT(DISTINCT message_id) * 1.0 / NULLIF(COUNT(DISTINCT conversation_id), 0), 2) as messages_per_conversation,
  
  -- Active space calculation (Option 2)
  COUNT(DISTINCT CASE WHEN hierarchy_level IN ('conversation', 'message') THEN space_id END) as active_spaces,
  ROUND(COUNT(DISTINCT CASE WHEN hierarchy_level IN ('conversation', 'message') THEN space_id END) * 100.0 / NULLIF(COUNT(DISTINCT space_id), 0), 2) as active_space_rate_pct,
  
  -- Error tracking
  COUNT(CASE WHEN status_code >= 400 THEN 1 END) as total_errors,
  ROUND(COUNT(CASE WHEN status_code >= 400 THEN 1 END) * 100.0 / COUNT(*), 2) as overall_error_rate_pct,
  
  -- Time range
  MIN(event_date) as first_event_date,
  MAX(event_date) as last_event_date,
  DATEDIFF(MAX(event_date), MIN(event_date)) + 1 as days_of_activity
  
FROM all_genie_actions
GROUP BY workspace_id

workspace_id,total_actions,unique_users,unique_sessions,unique_spaces,unique_conversations,unique_messages,space_level_actions,conversation_level_actions,message_level_actions,conversations_per_space,messages_per_conversation,active_spaces,active_space_rate_pct,total_errors,overall_error_rate_pct,first_event_date,last_event_date,days_of_activity
1516413757355523,693,2,473,6,14,29,340,49,304,2.33,2.07,4,66.67,60,8.66,2026-01-08,2026-01-29,22


In [0]:
-- Daily trends across all hierarchy levels
-- Use for time-series visualizations in dashboard (Option 3: Adoption Trends)

WITH all_genie_actions AS (
  SELECT 
    ae.workspace_id,
    ae.event_date,
    ae.user_identity.email as user_email,
    CAST(ae.request_params['space_id'] AS STRING) as space_id,
    CAST(ae.request_params['conversation_id'] AS STRING) as conversation_id,
    CAST(ae.request_params['message_id'] AS STRING) as message_id,
    ae.response.status_code as status_code,
    CASE 
      WHEN ae.request_params['message_id'] IS NOT NULL AND CAST(ae.request_params['message_id'] AS STRING) != '' THEN 'message'
      WHEN ae.request_params['conversation_id'] IS NOT NULL AND CAST(ae.request_params['conversation_id'] AS STRING) != '' THEN 'conversation'
      WHEN ae.request_params['space_id'] IS NOT NULL AND CAST(ae.request_params['space_id'] AS STRING) != '' THEN 'space'
      ELSE 'other'
    END as hierarchy_level
  FROM system.access.audit ae
  WHERE ae.service_name = 'aibiGenie'
    AND ae.request_params['space_id'] IS NOT NULL
    AND CAST(ae.request_params['space_id'] AS STRING) != ''
    AND ae.workspace_id = :workspace_id_filter
    AND ae.event_date >= date_sub(CURRENT_DATE(), :days_lookback)
    AND ae.event_date <= CURRENT_DATE()
)

SELECT 
  event_date,
  
  -- Daily action counts by level
  COUNT(CASE WHEN hierarchy_level = 'space' THEN 1 END) as space_actions,
  COUNT(CASE WHEN hierarchy_level = 'conversation' THEN 1 END) as conversation_actions,
  COUNT(CASE WHEN hierarchy_level = 'message' THEN 1 END) as message_actions,
  COUNT(*) as total_actions,
  
  -- Daily unique counts (Option 3: DAU)
  COUNT(DISTINCT user_email) as daily_active_users,
  COUNT(DISTINCT space_id) as active_spaces,
  COUNT(DISTINCT conversation_id) as active_conversations,
  COUNT(DISTINCT message_id) as messages_created,
  
  -- Daily error rate
  COUNT(CASE WHEN status_code >= 400 THEN 1 END) as errors,
  ROUND(COUNT(CASE WHEN status_code >= 400 THEN 1 END) * 100.0 / COUNT(*), 2) as error_rate_pct
  
FROM all_genie_actions
GROUP BY event_date
ORDER BY event_date DESC

event_date,space_actions,conversation_actions,message_actions,total_actions,daily_active_users,active_spaces,active_conversations,messages_created,errors,error_rate_pct
2026-01-29,93,10,124,227,2,3,9,22,15,6.61
2026-01-28,45,13,117,175,2,5,13,23,43,24.57
2026-01-27,141,21,51,213,2,4,4,7,2,0.94
2026-01-24,7,0,0,7,1,1,0,0,0,0.00
2026-01-23,46,5,12,63,1,1,1,1,0,0.00
2026-01-08,8,0,0,8,1,1,0,0,0,0.00


## Category 5: Message Details via Genie REST API

**Purpose:** Fetch actual message content and metadata using the Genie REST API.

**Why Needed:**
* `system.access.audit` only contains action metadata (who, when, what action)
* **Does NOT contain:** Message text, query SQL, space names, conversation titles
* **Solution:** Use Genie REST API to fetch detailed message content

**API Endpoint:**
```
GET {workspace_url}/api/2.0/genie/spaces/{space_id}/conversations/{conversation_id}/messages/{message_id}
```

**Authentication:** OAuth 2.0 using Service Principal

**Data Flow:**
1. Query `system.access.audit` for `genieGetConversationMessage` actions
2. Extract: workspace_url, space_id, conversation_id, message_id
3. Call Genie API for each message to get full details
4. Store results in Delta table for analysis

**Message Details Available:**
* Message content (user question)
* Generated SQL query
* Query results
* Attachments
* Feedback
* Timestamps

In [ ]:
# =============================================================================
# Config: Data extraction and secrets (code-based; no widgets)
# =============================================================================
# Edit these values for message extraction (Category 5). Dashboard (Categories 1–4) uses widgets.
# =============================================================================

# Secret scope for OAuth SP. Store keys: sp_client_id, sp_client_secret.
SECRETS_SCOPE = "genie-obs"

# Workspace(s) to extract messages from (required).
WORKSPACE_IDS = ["984752964297111", "1516413757355523"]

# Optional: restrict to specific Genie space(s). None = all spaces in the workspace(s).
SPACE_IDS = None  # e.g. None or ["01f01c91f1f414d59daaefd2b7ec82ea"]

# Days of audit data to consider for extraction.
DAYS_LOOKBACK_EXTRACTION = 30

# Optional: max messages to fetch per run (None = no limit).
EXTRACTION_MESSAGE_LIMIT = 100  # set to None for full extraction

print("✅ Data extraction config:")
print(f"   SECRETS_SCOPE = {SECRETS_SCOPE}")
print(f"   WORKSPACE_IDS = {WORKSPACE_IDS}")
print(f"   SPACE_IDS = {SPACE_IDS}")
print(f"   DAYS_LOOKBACK_EXTRACTION = {DAYS_LOOKBACK_EXTRACTION}")
print(f"   EXTRACTION_MESSAGE_LIMIT = {EXTRACTION_MESSAGE_LIMIT}")

In [0]:
%python
# =============================================================================
# OAuth Setup: Secrets + per-workspace tokens
# =============================================================================
# Uses SECRETS_SCOPE and WORKSPACE_IDS from Config cell. Builds WORKSPACE_HEADERS for Category 5.
# =============================================================================

import requests

# --- Load credentials from Databricks Secrets (scope from Config cell) ---
DATABRICKS_SP_CLIENT_ID = dbutils.secrets.get(scope=SECRETS_SCOPE, key="sp_client_id")
DATABRICKS_SP_CLIENT_SECRET = dbutils.secrets.get(scope=SECRETS_SCOPE, key="sp_client_secret")

# --- Get workspace URL for each WORKSPACE_IDS ---
workspace_ids_sql = ",".join([f"'{w}'" for w in WORKSPACE_IDS])
workspace_url_df = spark.sql(f"""
    SELECT workspace_id, workspace_url
    FROM system.access.workspaces_latest
    WHERE workspace_id IN ({workspace_ids_sql})
""")

WORKSPACE_HEADERS = {}
for row in workspace_url_df.collect():
    ws_id = row['workspace_id']
    url = (row['workspace_url'] or '').rstrip('/')
    if not url:
        continue
    token_url = f"{url}/oidc/v1/token"
    resp = requests.post(
        token_url,
        data={"grant_type": "client_credentials", "scope": "all-apis"},
        auth=(DATABRICKS_SP_CLIENT_ID, DATABRICKS_SP_CLIENT_SECRET),
        timeout=30
    )
    resp.raise_for_status()
    token = resp.json().get('access_token')
    WORKSPACE_HEADERS[ws_id] = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }

# For cells that still reference a single workspace_url or HEADERS (e.g. 5.4, 5.5)
workspace_url = workspace_url_df.collect()[0]['workspace_url'].rstrip('/') if workspace_url_df.count() > 0 else None
HEADERS = next(iter(WORKSPACE_HEADERS.values())) if WORKSPACE_HEADERS else {}

print("✅ OAuth loaded from secrets (code-based):")
print(f"   Scope: {SECRETS_SCOPE}")
print(f"   Workspaces with token: {list(WORKSPACE_HEADERS.keys())}")

✅ OAuth configuration loaded:
   Service Principal: 40ed6418-a18c-403d-8137-254f4489df49
   Workspace URL: https://adb-984752964297111.11.azuredatabricks.net
   Token URL: https://adb-984752964297111.11.azuredatabricks.net/oidc/v1/token


In [0]:
%python
# =============================================================================
# OAuth Token Validation
# =============================================================================
# Purpose: 
#   1. Authenticate with Databricks using Service Principal OAuth 2.0
#   2. Obtain workspace-level access token
#   3. Set up HTTP headers for Genie API calls
# =============================================================================

import requests
import json
from datetime import datetime, timedelta

# --- Validation: WORKSPACE_HEADERS built in previous cell ---
if not WORKSPACE_HEADERS:
    raise Exception("❌ No workspace tokens. Check SECRETS_SCOPE and WORKSPACE_IDS in Config cell.")

print("✅ OAuth authentication complete")
print(f"   Tokens ready for {len(WORKSPACE_HEADERS)} workspace(s)")
print(f"   Use WORKSPACE_HEADERS[workspace_id] in API calls")

🔐 Authenticating with workspace...
✅ Workspace authentication successful (status: 200)
   Token will be used for Genie API calls

✅ OAuth authentication complete
   Workspace token: ✅
   HTTP headers configured for API calls


In [0]:
%python
# =============================================================================
# Extract Unique Messages for API Enrichment
# =============================================================================
# Purpose: Get DISTINCT messages from audit logs (deduplicated)
# Note: Same message_id can appear multiple times in audit logs
# =============================================================================

# Use code-based config (Config cell)
workspace_ids_sql = ",".join([f"'{w}'" for w in WORKSPACE_IDS])
space_filter_sql = ""
if SPACE_IDS:
    space_ids_esc = ",".join([f"'{s}'" for s in SPACE_IDS])
    space_filter_sql = f"    AND CAST(ae.request_params['space_id'] AS STRING) IN ({space_ids_esc})\n"

limit_sql = f"\nLIMIT {EXTRACTION_MESSAGE_LIMIT}" if EXTRACTION_MESSAGE_LIMIT else ""

print(f"🔍 Querying unique messages from system.access.audit...")
print(f"   Workspaces: {WORKSPACE_IDS}")
print(f"   Space filter: {SPACE_IDS or 'all'}")
print(f"   Days lookback: {DAYS_LOOKBACK_EXTRACTION}")

# Query DISTINCT messages (deduplicated by message_id)
# Using parameter values in SQL (properly escaped)
messages_for_api = spark.sql(f"""
WITH ranked_messages AS (
  SELECT 
    -- Workspace context
    w.workspace_name,
    w.workspace_url,
    ae.workspace_id,
    
    -- Genie hierarchy (all required for API call)
    CAST(ae.request_params['space_id'] AS STRING) as space_id,
    CAST(ae.request_params['conversation_id'] AS STRING) as conversation_id,
    CAST(ae.request_params['message_id'] AS STRING) as message_id,
    
    -- Action metadata
    ae.action_name,
    ae.event_time,
    ae.event_date,
    ae.user_identity.email as user_email,
    ae.response.status_code as status_code,
    
    -- Deduplication: Keep most recent access per message
    ROW_NUMBER() OVER (
      PARTITION BY ae.workspace_id, 
                   CAST(ae.request_params['message_id'] AS STRING)
      ORDER BY ae.event_time DESC
    ) as rn
    
  FROM system.access.audit ae
  
  -- Join workspace metadata for workspace_url
  INNER JOIN system.access.workspaces_latest w
    ON ae.workspace_id = w.workspace_id
  
  WHERE ae.service_name = 'aibiGenie'
    -- Filter for message retrieval actions
    AND ae.action_name IN ('genieGetConversationMessage', 'getConversationMessage')
    -- Ensure all required IDs are present and not empty
    AND ae.request_params['space_id'] IS NOT NULL
    AND CAST(ae.request_params['space_id'] AS STRING) != ''
    AND ae.request_params['conversation_id'] IS NOT NULL
    AND CAST(ae.request_params['conversation_id'] AS STRING) != ''
    AND ae.request_params['message_id'] IS NOT NULL
    AND CAST(ae.request_params['message_id'] AS STRING) != ''
    -- Only successful API calls
    AND ae.response.status_code = 200
    -- Parameter filters (from Config: WORKSPACE_IDS, optional SPACE_IDS)
    AND ae.workspace_id IN ({workspace_ids_sql})
    AND ae.event_date >= date_sub(CURRENT_DATE(), {DAYS_LOOKBACK_EXTRACTION})
    AND ae.event_date <= CURRENT_DATE()
    {space_filter_sql}
)

SELECT 
  workspace_name,
  workspace_url,
  workspace_id,
  space_id,
  conversation_id,
  message_id,
  action_name,
  event_time as last_accessed_time,
  event_date,
  user_email,
  status_code,
  CONCAT(
    workspace_url,
    '/api/2.0/genie/spaces/', space_id,
    '/conversations/', conversation_id,
    '/messages/', message_id
  ) as api_url
FROM ranked_messages
WHERE rn = 1  -- Keep only one record per message (most recent)
ORDER BY last_accessed_time DESC
{limit_sql}
""")

# Create temp view for SQL access
messages_for_api.createOrReplaceTempView("messages_for_api")

total_count = messages_for_api.count()
unique_count = messages_for_api.select('message_id').distinct().count()

print(f"\n✅ Named DataFrame created: messages_for_api")
print(f"   Total messages: {total_count}")
print(f"   Unique messages: {unique_count}")
print(f"   Deduplication: {'✅ CLEAN' if total_count == unique_count else '⚠️ DUPLICATES FOUND'}")
print(f"   Columns: {', '.join(messages_for_api.columns)}")

# Display sample
print(f"\n📋 Sample Messages (Deduplicated):")
display(messages_for_api.select(
    'workspace_name',
    'workspace_id',
    'space_id',
    'conversation_id', 
    'message_id',
    'last_accessed_time'
).limit(10))

🔍 Querying unique messages from system.access.audit...
   Workspace: 984752964297111
   Days lookback: 30

✅ Named DataFrame created: messages_for_api
   Total messages: 100
   Unique messages: 100
   Deduplication: ✅ CLEAN
   Columns: workspace_name, workspace_url, workspace_id, space_id, conversation_id, message_id, action_name, last_accessed_time, event_date, user_email, status_code, api_url

📋 Sample Messages (Deduplicated):


workspace_name,workspace_id,space_id,conversation_id,message_id,last_accessed_time
field-eng-east,984752964297111,01f01c91f1f414d59daaefd2b7ec82ea,01f0fefe631f1f38811b05300694886d,01f0fefe65a21b9eba48a6c65c59b294,2026-01-31T23:41:48.170Z
field-eng-east,984752964297111,01f01c91f1f414d59daaefd2b7ec82ea,01f0fefe631f1f38811b05300694886d,01f0fefe63291c349d1ef6fb91aedfd7,2026-01-31T23:41:42.733Z
field-eng-east,984752964297111,01f0c482e842191587af6a40ad4044d8,01f0fefe41af1daf9de177de6dab9308,01f0fefe48ec11958761bd487567ad4b,2026-01-31T23:41:00.592Z
field-eng-east,984752964297111,01f0c482e842191587af6a40ad4044d8,01f0fefe41af1daf9de177de6dab9308,01f0fefe45041ac9883048cf6a93957b,2026-01-31T23:40:54.059Z
field-eng-east,984752964297111,01f0c482e842191587af6a40ad4044d8,01f0fefe41af1daf9de177de6dab9308,01f0fefe41b81c17b5159910292916c6,2026-01-31T23:40:47.286Z
field-eng-east,984752964297111,01f0fd8c745e13bb8df621f651d596b9,01f0fef1f7ec1472b27c1c01a8eff1e8,01f0fef1f7fa1d2f894f945f5711d636,2026-01-31T22:12:55.571Z
field-eng-east,984752964297111,01f0eac186d11b9897bc1d43836cc4e1,01f0fee345771d78b180da3450084e83,01f0fee3458015fbbccde144985f4418,2026-01-31T20:27:37.707Z
field-eng-east,984752964297111,01f0eababd9f1bcab5dea65cf67e48e3,01f0fee34142175d9de9438b1454b259,01f0fee3414b1749a7d9b0cb47df4785,2026-01-31T20:27:29.545Z
field-eng-east,984752964297111,01f0eab621401f9faa11e680f5a2bcd0,01f0fee33c0c17cca8f40ec16c08c7ee,01f0fee33c1511398f480f73a583e2b4,2026-01-31T20:27:23.027Z
field-eng-east,984752964297111,01f096815e8b1189b3d4cb4922764481,01f0fee183071c9ca926f150365fb985,01f0fee183411c20b129d5f76b110638,2026-01-31T20:14:59.969Z


In [0]:
%python
# =============================================================================
# Fetch Message Details from Genie API & Extract SQL from Attachments
# =============================================================================
# Purpose: 
#   1. Fetch message details for unique messages from cell 5.1
#   2. Extract SQL, statement_id, and description from query attachments
#   3. Store in temp view for analysis (validation before persistence)
# =============================================================================

import requests
import json
import time
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, LongType
from pyspark.sql.functions import current_timestamp
from datetime import datetime

# Configuration
SAMPLE_SIZE = 50  # Number of messages to fetch for validation

# --- Get Messages from Named DataFrame ---
print("📊 Getting unique messages from messages_for_api...")
try:
    messages_df = spark.table("messages_for_api")  # Use deduplicated view from cell 5.1
    messages_list = messages_df.limit(SAMPLE_SIZE).collect()
    print(f"   Found {len(messages_list)} unique messages for API enrichment")
except Exception as e:
    print(f"⚠️  Error loading messages_for_api: {str(e)}")
    print("   Run cell 5.1 first to create the DataFrame.")
    raise

if len(messages_list) == 0:
    print("⚠️  No messages found. Adjust filters in cell 5.1.")
else:
    # --- Fetch Message Details from API ---
    print(f"\n🚀 Fetching {len(messages_list)} message(s) from Genie API...\n")
    
    message_details = []
    stats = {'success': 0, 'failed': 0, 'auth_error': 0, 'with_sql': 0, 'with_statement_id': 0}
    
    for idx, msg in enumerate(messages_list, 1):
        try:
            # Access Row fields using bracket notation
            workspace_id = msg['workspace_id']
            workspace_name = msg['workspace_name']
            space_id = msg['space_id']
            conversation_id = msg['conversation_id']
            message_id = msg['message_id']
            
            api_url = msg['api_url']
            headers = WORKSPACE_HEADERS.get(msg['workspace_id'])
            if not headers:
                stats['failed'] += 1
                continue
            # Call Genie API
            response = requests.get(api_url, headers=headers, timeout=10)
            
            if response.status_code == 200:
                msg_data = response.json()
                
                # Extract statement_id, query_sql, and description from first query attachment
                statement_id = None
                query_sql = None
                query_description = None
                attachments = msg_data.get('attachments', [])
                
                for attachment in attachments:
                    if 'query' in attachment:
                        query_obj = attachment['query']
                        if query_obj.get('statement_id'):
                            statement_id = query_obj['statement_id']
                            query_sql = query_obj.get('query', '')
                            query_description = query_obj.get('description', '')
                            break  # Use first query attachment
                
                # Track extraction stats
                if query_sql:
                    stats['with_sql'] += 1
                if statement_id:
                    stats['with_statement_id'] += 1
                
                # Extract key fields from API response
                message_details.append({
                    'workspace_id': workspace_id,
                    'workspace_name': workspace_name,
                    'space_id': space_id,
                    'conversation_id': conversation_id,
                    'message_id': message_id,
                    'content': msg_data.get('content', ''),
                    'status': msg_data.get('status', ''),
                    'created_timestamp': msg_data.get('created_timestamp'),
                    'updated_timestamp': msg_data.get('last_updated_timestamp'),
                    'query_sql': query_sql,  # Extracted from attachments
                    'query_description': query_description,  # What Genie understood
                    'query_status': msg_data.get('query', {}).get('status', '') if msg_data.get('query') else '',
                    'statement_id': statement_id,  # Links to system.query.history
                    'attachments_count': len(attachments),
                    'api_fetch_status': 'SUCCESS',
                    'api_fetch_error': None
                })
                stats['success'] += 1
                if idx <= 10 or idx % 10 == 0:  # Show first 10, then every 10th
                    stmt_preview = statement_id[:12] if statement_id else 'N/A'
                    sql_status = '✅ YES' if query_sql else '❌ NO'
                    print(f"   [{idx}/{len(messages_list)}] ✅ {message_id[:12]}... SQL:{sql_status} stmt:{stmt_preview}...")
                
            elif response.status_code in [401, 403]:
                stats['auth_error'] += 1
                print(f"   [{idx}/{len(messages_list)}] 🔒 Auth error (status {response.status_code}): {message_id[:12]}...")
                message_details.append({
                    'workspace_id': workspace_id,
                    'workspace_name': workspace_name,
                    'space_id': space_id,
                    'conversation_id': conversation_id,
                    'message_id': message_id,
                    'content': None,
                    'status': None,
                    'created_timestamp': None,
                    'updated_timestamp': None,
                    'query_sql': None,
                    'query_description': None,
                    'query_status': None,
                    'statement_id': None,
                    'attachments_count': None,
                    'api_fetch_status': 'AUTH_ERROR',
                    'api_fetch_error': f"HTTP {response.status_code}: {response.text[:200]}"
                })
            else:
                stats['failed'] += 1
                if idx <= 10:  # Only show first 10 failures
                    print(f"   [{idx}/{len(messages_list)}] ❌ Failed (status {response.status_code}): {message_id[:12]}...")
                message_details.append({
                    'workspace_id': workspace_id,
                    'workspace_name': workspace_name,
                    'space_id': space_id,
                    'conversation_id': conversation_id,
                    'message_id': message_id,
                    'content': None,
                    'status': None,
                    'created_timestamp': None,
                    'updated_timestamp': None,
                    'query_sql': None,
                    'query_description': None,
                    'query_status': None,
                    'statement_id': None,
                    'attachments_count': None,
                    'api_fetch_status': 'FAILED',
                    'api_fetch_error': f"HTTP {response.status_code}: {response.text[:200]}"
                })
            
            # Rate limiting
            time.sleep(0.2)
            
        except Exception as e:
            stats['failed'] += 1
            if idx <= 10:
                print(f"   [{idx}/{len(messages_list)}] ❌ Error: {str(e)[:100]}")
            message_details.append({
                'workspace_id': workspace_id if 'workspace_id' in locals() else None,
                'workspace_name': workspace_name if 'workspace_name' in locals() else None,
                'space_id': space_id if 'space_id' in locals() else None,
                'conversation_id': conversation_id if 'conversation_id' in locals() else None,
                'message_id': message_id if 'message_id' in locals() else None,
                'content': None,
                'status': None,
                'created_timestamp': None,
                'updated_timestamp': None,
                'query_sql': None,
                'query_description': None,
                'query_status': None,
                'statement_id': None,
                'attachments_count': None,
                'api_fetch_status': 'ERROR',
                'api_fetch_error': str(e)[:500]
            })
    
    # --- Create DataFrame and Temp View ---
    if len(message_details) > 0:
        schema = StructType([
            StructField("workspace_id", StringType(), True),
            StructField("workspace_name", StringType(), True),
            StructField("space_id", StringType(), True),
            StructField("conversation_id", StringType(), True),
            StructField("message_id", StringType(), True),
            StructField("content", StringType(), True),
            StructField("status", StringType(), True),
            StructField("created_timestamp", LongType(), True),
            StructField("updated_timestamp", LongType(), True),
            StructField("query_sql", StringType(), True),
            StructField("query_description", StringType(), True),
            StructField("query_status", StringType(), True),
            StructField("statement_id", StringType(), True),
            StructField("attachments_count", IntegerType(), True),
            StructField("api_fetch_status", StringType(), True),
            StructField("api_fetch_error", StringType(), True)
        ])
        
        message_details_df = spark.createDataFrame(message_details, schema=schema)
        
        # Create temporary view
        message_details_df.createOrReplaceTempView("genie_message_details_validation")
        
        # Verify no duplicates in final dataset
        total_rows = message_details_df.count()
        unique_messages = message_details_df.select('message_id').distinct().count()
        
        # Display statistics
        print(f"\n📊 API Fetch Results:")
        print(f"   ✅ Success: {stats['success']}")
        print(f"   🔒 Auth Errors: {stats['auth_error']}")
        print(f"   ❌ Failed: {stats['failed']}")
        print(f"   Total: {len(message_details)}")
        
        print(f"\n🔗 Data Extraction Quality:")
        print(f"   Messages with statement_id: {stats['with_statement_id']}/{stats['success']} ({stats['with_statement_id']/stats['success']*100:.1f}%)")
        print(f"   Messages with query_sql: {stats['with_sql']}/{stats['success']} ({stats['with_sql']/stats['success']*100:.1f}%)")
        print(f"   Can join with system.query.history: {'YES ✅' if stats['with_statement_id'] > 0 else 'NO ❌'}")
        
        print(f"\n🛡️ Deduplication Check:")
        print(f"   Total rows: {total_rows}")
        print(f"   Unique messages: {unique_messages}")
        print(f"   Status: {'✅ CLEAN - No duplicates' if total_rows == unique_messages else '⚠️ DUPLICATES FOUND'}")
        
        success_rate = (stats['success'] / len(message_details) * 100) if len(message_details) > 0 else 0
        print(f"\n   API Success Rate: {success_rate:.1f}%")
        print(f"\n✅ Temporary view created: genie_message_details_validation")
        
        # Display sample results
        print(f"\n📋 Sample Results:")
        display(message_details_df.select(
            'workspace_name',
            'message_id',
            'content',
            'status',
            'statement_id',
            'attachments_count',
            'api_fetch_status'
        ).limit(10))
        
        # Show messages with SQL
        if stats['with_sql'] > 0:
            print(f"\n📝 Messages with Generated SQL ({stats['with_sql']} messages):")
            display(message_details_df.filter("query_sql IS NOT NULL AND query_sql != ''").select(
                'message_id',
                'content',
                'query_description',
                'statement_id'
            ).limit(5))
        
        # Show any errors
        errors_df = message_details_df.filter("api_fetch_status != 'SUCCESS'")
        if errors_df.count() > 0:
            print(f"\n⚠️  Errors Found:")
            display(errors_df.select('message_id', 'api_fetch_status', 'api_fetch_error').limit(5))
        
        print(f"\n💡 Next Steps:")
        print(f"   1. Run cell 5.3 to see complete user journey with query performance")
        print(f"   2. Run cell 5.5 to see summary analytics")
        print(f"   3. Run cell 5.6 to persist to Delta table (optional)")
        
    else:
        print("\n⚠️  No message details fetched")

📊 Getting unique messages from messages_for_api...
   Found 15 unique messages for API enrichment

🚀 Fetching 15 message(s) from Genie API...

   [1/15] ✅ 01f0fd344485... SQL:❌ NO stmt:N/A...
   [2/15] ✅ 01f0fd344afc... SQL:✅ YES stmt:01f0fd34-4d8...
   [3/15] ✅ 01f0fd346586... SQL:✅ YES stmt:01f0fd34-749...
   [4/15] ✅ 01f0fd345bec... SQL:❌ NO stmt:N/A...
   [5/15] ✅ 01f0fd3417c8... SQL:✅ YES stmt:01f0fd34-2fb...
   [6/15] ✅ 01f0fd34314c... SQL:✅ YES stmt:01f0fd34-3ac...
   [7/15] ✅ 01f0f89c5e56... SQL:✅ YES stmt:01f0f89c-636...
   [8/15] ✅ 01f07c6cdada... SQL:❌ NO stmt:N/A...
   [9/15] ✅ 01f0fb2a89b1... SQL:✅ YES stmt:01f0fb2a-974...
   [10/15] ✅ 01f0aab49ed7... SQL:✅ YES stmt:01f0aab4-a16...

📊 API Fetch Results:
   ✅ Success: 15
   🔒 Auth Errors: 0
   ❌ Failed: 0
   Total: 15

🔗 Data Extraction Quality:
   Messages with statement_id: 11/15 (73.3%)
   Messages with query_sql: 11/15 (73.3%)
   Can join with system.query.history: YES ✅

🛡️ Deduplication Check:
   Total rows: 15
   Uni

workspace_name,message_id,content,status,statement_id,attachments_count,api_fetch_status
DBW-BK-wx1,01f0fd3444851cf3be2a1048e5287172,Are Tier 2 Suppliers a Bottleneck for High-Demand Tier 1 Components?,COMPLETED,null,2,SUCCESS
DBW-BK-wx1,01f0fd344afc14158579397bd3cacb76,What is the total quantity needed for each part based on the sales forecast?,COMPLETED,01f0fd34-4d8b-1472-8da0-4eed2fab7120,3,SUCCESS
DBW-BK-wx1,01f0fd3465861986a917b3d94ac7d6f4,go for it,COMPLETED,01f0fd34-749b-1deb-93fa-e13ffe8b2765,4,SUCCESS
DBW-BK-wx1,01f0fd345bec143f83c0f72c9a55583d,yes,COMPLETED,null,1,SUCCESS
DBW-BK-wx1,01f0fd3417c81620b1f043a30a77ddae,What is the total quantity needed for each part in the next 3 months based on the sales forecast?,FAILED,01f0fd34-2fb3-1414-aaaa-4bbb3dc175c0,3,SUCCESS
DBW-BK-wx1,01f0fd34314c1865834781b40658ae24,Can our supply base scale effectively to meet sales forecast?,COMPLETED,01f0fd34-3ace-1611-8013-52fc8b07ea03,4,SUCCESS
DBW-BK-wx1,01f0f89c5e561bc4948986d36acfe2a3,What is the total quantity needed for each part based on the sales forecast?,COMPLETED,01f0f89c-6368-1258-8ef8-276f53c29c56,2,SUCCESS
DBW-BK-wx1,01f07c6cdada159883cdd8a8882488a5,"I will provide you a chat history, where your name is agent-new-space. Please help with the described information in the chat history. None:",COMPLETED,null,1,SUCCESS
DBW-BK-wx1,01f0fb2a89b11db196f34f7f13704ca1,What is the total quantity needed for each part in the next 6 months based on the sales forecast?,COMPLETED,01f0fb2a-9749-1e3b-b395-94e943b7c1e8,3,SUCCESS
DBW-BK-wx1,01f0aab49ed71edd97f86c4ac9275ad5,What is the total quantity needed for each part?,COMPLETED,01f0aab4-a168-1c54-a95b-51185c197142,2,SUCCESS



📝 Messages with Generated SQL (11 messages):


message_id,content,query_description,statement_id
01f0fd344afc14158579397bd3cacb76,What is the total quantity needed for each part based on the sales forecast?,You want to see the total quantity needed for each part based on the sales forecast for all vehicle programs.,01f0fd34-4d8b-1472-8da0-4eed2fab7120
01f0fd3465861986a917b3d94ac7d6f4,go for it,You want to see the total quantity needed for each part based on the sales forecast for the next 3 months.,01f0fd34-749b-1deb-93fa-e13ffe8b2765
01f0fd3417c81620b1f043a30a77ddae,What is the total quantity needed for each part in the next 3 months based on the sales forecast?,You want to see the total quantity needed for each part over the next three months based on the sales forecast.,01f0fd34-2fb3-1414-aaaa-4bbb3dc175c0
01f0fd34314c1865834781b40658ae24,Can our supply base scale effectively to meet sales forecast?,You want to see if the supply base can meet the sales forecast for the next three months by comparing the total quantity of each part needed based on the sales forecast with the total quantity sourced from suppliers during the same period.,01f0fd34-3ace-1611-8013-52fc8b07ea03
01f0f89c5e561bc4948986d36acfe2a3,What is the total quantity needed for each part based on the sales forecast?,"You want to see the total quantity needed for each part based on the sales forecast, including the part number and part name, ordered by the total quantity needed from highest to lowest.",01f0f89c-6368-1258-8ef8-276f53c29c56



💡 Next Steps:
   1. Run cell 5.3 to see complete user journey with query performance
   2. Run cell 5.5 to see summary analytics
   3. Run cell 5.6 to persist to Delta table (optional)


In [0]:
-- =============================================================================
-- Query: Complete Message Flow with Query Performance
-- =============================================================================
-- Purpose: Show end-to-end flow from user question → SQL → execution → results
-- Joins: genie_message_details_validation + system.query.history
-- =============================================================================

WITH message_base AS (
  -- Get message details from temp view
  SELECT 
    workspace_id,
    workspace_name,
    space_id,
    conversation_id,
    message_id,
    content as user_question,
    status as message_status,
    created_timestamp,
    query_sql as generated_sql,
    query_description,
    query_status as sql_generation_status,
    statement_id,
    attachments_count
  FROM genie_message_details_validation
  WHERE api_fetch_status = 'SUCCESS'
),

query_performance AS (
  -- Get query execution details from system.query.history
  SELECT 
    qh.statement_id,
    qh.workspace_id,
    qh.start_time as query_start_time,
    qh.end_time as query_end_time,
    qh.execution_status as query_execution_status,
    qh.total_duration_ms,
    qh.read_rows,
    qh.written_rows,
    qh.read_bytes,
    qh.compute.warehouse_id as execution_warehouse,
    qh.error_message as query_error_message
  FROM system.query.history qh
  WHERE qh.workspace_id = :workspace_id_filter
    AND DATE(qh.start_time) >= date_sub(CURRENT_DATE(), :days_lookback)
)

SELECT 
  -- Message ID
  mb.message_id,
  
  -- 📝 STEP 1: USER INPUT
  mb.user_question,
  FROM_UNIXTIME(mb.created_timestamp / 1000) as message_created_at,
  
  -- 🤖 STEP 2: GENIE PROCESSING
  mb.message_status,
  mb.attachments_count as context_attachments,
  
  -- 💾 STEP 3: SQL GENERATION (from attachments)
  mb.query_description as what_genie_understood,
  CASE 
    WHEN mb.generated_sql IS NOT NULL AND mb.generated_sql != '' 
    THEN CONCAT(SUBSTRING(mb.generated_sql, 1, 150), '...')
    ELSE 'No SQL generated'
  END as sql_preview,
  mb.sql_generation_status,
  
  -- ⚡ STEP 4: QUERY EXECUTION
  qp.statement_id,
  qp.query_execution_status,
  qp.execution_warehouse,
  qp.query_start_time,
  
  -- 📊 STEP 5: PERFORMANCE
  ROUND(qp.total_duration_ms / 1000.0, 2) as duration_seconds,
  qp.read_rows,
  qp.written_rows,
  ROUND(qp.read_bytes / 1024.0 / 1024.0, 2) as read_mb,
  
  -- ✅ STEP 6: OUTCOME
  CASE 
    WHEN mb.message_status = 'COMPLETED' AND qp.query_execution_status = 'FINISHED' THEN 'SUCCESS'
    WHEN mb.message_status = 'FAILED' OR qp.query_execution_status = 'FAILED' THEN 'FAILED'
    WHEN qp.query_execution_status = 'CANCELED' THEN 'CANCELED'
    ELSE 'UNKNOWN'
  END as overall_status,
  qp.query_error_message,
  
  -- ⏱️ TIMING
  TIMESTAMPDIFF(SECOND, FROM_UNIXTIME(mb.created_timestamp / 1000), qp.query_start_time) as latency_seconds
  
FROM message_base mb
LEFT JOIN query_performance qp
  ON mb.statement_id = qp.statement_id
  AND mb.workspace_id = qp.workspace_id
  
ORDER BY mb.created_timestamp DESC
LIMIT 20;

message_id,user_question,message_created_at,message_status,context_attachments,what_genie_understood,sql_preview,sql_generation_status,statement_id,query_execution_status,execution_warehouse,query_start_time,duration_seconds,read_rows,written_rows,read_mb,overall_status,query_error_message,latency_seconds
01f0fd3465861986a917b3d94ac7d6f4,go for it,2026-01-29 17:03:13,COMPLETED,4,You want to see the total quantity needed for each part based on the sales forecast for the next 3 months.,"SELECT pm.part_number, pm.part_name, SUM(sf.vehicle_quantity * bom.quantity_used) AS total_quantity_needed FROM `bk`.`supply_chain_sop`.`sales_forecas...",,01f0fd34-749b-1deb-93fa-e13ffe8b2765,FINISHED,093d4ec27ed4bdee,2026-01-29T17:03:38.503Z,0.57,0,0,0.00,SUCCESS,null,25
01f0fd345bec143f83c0f72c9a55583d,yes,2026-01-29 17:02:57,COMPLETED,1,null,No SQL generated,,null,null,null,null,null,null,null,null,UNKNOWN,null,null
01f0fd344afc14158579397bd3cacb76,What is the total quantity needed for each part based on the sales forecast?,2026-01-29 17:02:28,COMPLETED,3,You want to see the total quantity needed for each part based on the sales forecast for all vehicle programs.,"SELECT pm.part_number, pm.part_name, SUM(sf.vehicle_quantity * bom.quantity_used) AS total_quantity_needed FROM `bk`.`supply_chain_sop`.`sales_forecas...",,01f0fd34-4d8b-1472-8da0-4eed2fab7120,FINISHED,093d4ec27ed4bdee,2026-01-29T17:02:32.963Z,3.31,32,0,0.00,SUCCESS,null,4
01f0fd3444851cf3be2a1048e5287172,Are Tier 2 Suppliers a Bottleneck for High-Demand Tier 1 Components?,2026-01-29 17:02:17,COMPLETED,2,null,No SQL generated,,null,null,null,null,null,null,null,null,UNKNOWN,null,null
01f0fd34314c1865834781b40658ae24,Can our supply base scale effectively to meet sales forecast?,2026-01-29 17:01:45,COMPLETED,4,You want to see if the supply base can meet the sales forecast for the next three months by comparing the total quantity of each part needed based on the sales forecast with the total quantity sourced from suppliers during the same period.,WITH next_three_months AS (SELECT DISTINCT forecast_month FROM `bk`.`supply_chain_sop`.`sales_forecasts` WHERE forecast_month >= CURRENT_DATE ORDER BY...,,01f0fd34-3ace-1611-8013-52fc8b07ea03,FINISHED,093d4ec27ed4bdee,2026-01-29T17:02:01.525Z,1.00,0,0,0.00,SUCCESS,null,16
01f0fd3417c81620b1f043a30a77ddae,What is the total quantity needed for each part in the next 3 months based on the sales forecast?,2026-01-29 17:01:02,FAILED,3,You want to see the total quantity needed for each part over the next three months based on the sales forecast.,WITH next_three_months AS (SELECT DISTINCT forecast_month FROM `bk`.`supply_chain_sop`.`sales_forecasts` WHERE forecast_month >= CURRENT_DATE ORDER BY...,,01f0fd34-2fb3-1414-aaaa-4bbb3dc175c0,FINISHED,093d4ec27ed4bdee,2026-01-29T17:01:42.895Z,0.48,0,0,0.00,FAILED,null,40
01f0fb2aa1711ed897acbd6d4358c5bc,What is the total quantity needed for each part in the next 3 months based on the sales forecast?,2026-01-27 02:48:16,COMPLETED,3,You want to see the total quantity needed for each part over the next three months based on the sales forecast.,WITH next_3_months AS (SELECT DISTINCT forecast_month FROM `bk`.`supply_chain_sop`.`sales_forecasts` ORDER BY forecast_month ASC LIMIT 3) SELECT pm.pa...,,01f0fb2a-a49c-1ab2-987e-accaa796a92c,FINISHED,093d4ec27ed4bdee,2026-01-27T02:48:21.746Z,1.48,35,0,0.01,SUCCESS,null,5
01f0fb2a89b11db196f34f7f13704ca1,What is the total quantity needed for each part in the next 6 months based on the sales forecast?,2026-01-27 02:47:36,COMPLETED,3,You want to see the total quantity needed for each part over the next six months based on the sales forecast.,WITH next_6_months AS (SELECT DISTINCT forecast_month FROM `bk`.`supply_chain_sop`.`sales_forecasts` ORDER BY forecast_month ASC LIMIT 6) SELECT pm.pa...,,01f0fb2a-9749-1e3b-b395-94e943b7c1e8,FINISHED,093d4ec27ed4bdee,2026-01-27T02:47:59.393Z,3.65,35,0,0.01,SUCCESS,null,23
01f0f89c5e561bc4948986d36acfe2a3,What is the total quantity neede

In [0]:
%python
# =============================================================================
# Extract Attachment Details from API Response
# =============================================================================
# Purpose: 
#   1. Fetch a message with attachments
#   2. Extract attachment_id, type, and metadata
#   3. Show what context is sent to the LLM
# =============================================================================

import requests
import json
from pprint import pprint

# Get a message with attachments from the temp view
print("📎 Finding messages with attachments...")
messages_with_attachments = spark.sql("""
    SELECT message_id, space_id, conversation_id, attachments_count
    FROM genie_message_details_validation
    WHERE attachments_count > 0
    ORDER BY attachments_count DESC
    LIMIT 1
""").collect()

if len(messages_with_attachments) == 0:
    print("⚠️  No messages with attachments found")
else:
    msg = messages_with_attachments[0]
    print(f"   Found message with {msg.attachments_count} attachment(s)")
    print(f"   Message ID: {msg.message_id}")
    
    # Fetch full message details from API
    api_url = f"{workspace_url}/api/2.0/genie/spaces/{msg.space_id}/conversations/{msg.conversation_id}/messages/{msg.message_id}"
    
    print(f"\n🔍 Fetching full message details from API...")
    response = requests.get(api_url, headers=HEADERS, timeout=10)
    
    if response.status_code == 200:
        msg_data = response.json()
        
        print(f"\n✅ Message Details:")
        print(f"   Content: {msg_data.get('content', 'N/A')}")
        print(f"   Status: {msg_data.get('status', 'N/A')}")
        print(f"   Created: {msg_data.get('created_timestamp', 'N/A')}")
        
        # Extract attachments
        attachments = msg_data.get('attachments', [])
        print(f"\n📎 Attachments ({len(attachments)}):")
        print(f"   These attachments provide context to the LLM\n")
        
        for idx, attachment in enumerate(attachments, 1):
            print(f"   ─── Attachment {idx} ───")
            print(f"   attachment_id: {attachment.get('attachment_id', 'N/A')}")
            
            # Query attachment details
            query = attachment.get('query', {})
            if query:
                print(f"   type: QUERY")
                print(f"   query_details:")
                print(f"      statement_id: {query.get('statement_id', 'N/A')}")
                print(f"      description: {query.get('description', 'N/A')[:100]}..." if query.get('description') else "      description: N/A")
                query_text = query.get('query', '')
                if query_text:
                    print(f"      query_sql: {query_text[:150]}...")
                else:
                    print(f"      query_sql: N/A")
            
            # Text content (if any)
            text = attachment.get('text', {})
            if text:
                print(f"   type: TEXT")
                print(f"   text_content:")
                content = text.get('content', '')
                if content:
                    print(f"      {content[:200]}...")
                else:
                    print(f"      N/A")
            
            # Chart/visualization (if any)
            chart = attachment.get('chart', {})
            if chart:
                print(f"   type: CHART")
                print(f"   chart_type: {chart.get('type', 'N/A')}")
            
            print()  # Blank line between attachments
        
        # Show full attachment structure for first attachment
        if len(attachments) > 0:
            print(f"\n📋 Full JSON Structure of First Attachment:")
            print(json.dumps(attachments[0], indent=2)[:1000] + "...")
        
        print(f"\n🔑 Available Fields in Message API Response:")
        print(f"   {', '.join(msg_data.keys())}")
        
        print(f"\n💡 Key Insights:")
        print(f"   • attachment_id: Unique identifier for each attachment (use for tracking)")
        print(f"   • query: Contains SQL query, statement_id, description sent to LLM")
        print(f"   • text: Contains text/instructions sent to LLM as context")
        print(f"   • chart: Contains visualization metadata (if applicable)")
        print(f"   • statement_id: Links to system.query.history for query performance")
        print(f"\n   ➡️  Use attachment_id to track what context was provided to the LLM")
        print(f"   ➡️  Use statement_id to join with system.query.history for query metrics")
        
    else:
        print(f"❌ Failed to fetch message (status {response.status_code})")
        print(f"   Error: {response.text[:200]}")

📎 Finding messages with attachments...
   Found message with 4 attachment(s)
   Message ID: 01f0fd3465861986a917b3d94ac7d6f4

🔍 Fetching full message details from API...

✅ Message Details:
   Content: go for it
   Status: COMPLETED
   Created: 1769706193193

📎 Attachments (4):
   These attachments provide context to the LLM

   ─── Attachment 1 ───
   attachment_id: 01f0fd3474941363af76060e4c19a9a8
   type: QUERY
   query_details:
      statement_id: 01f0fd34-749b-1deb-93fa-e13ffe8b2765
      description: You want to see the total quantity needed for each part based on the sales forecast for the next 3 m...
      query_sql: SELECT pm.part_number, pm.part_name, SUM(sf.vehicle_quantity * bom.quantity_used) AS total_quantity_needed FROM `bk`.`supply_chain_sop`.`sales_forecas...

   ─── Attachment 2 ───
   attachment_id: 01f0fd34756e17458bd480680259cdd4
   type: TEXT
   text_content:
      Would you prefer to see the total quantity needed for each part based on the sales forecast using ca

In [0]:
%python
# =============================================================================
# Demonstrate Proper SQL Extraction from Attachments
# =============================================================================
# Purpose: Show how to extract the complete context and SQL from API response
# Goal 1: Capture space_id, conversation_id, message_id → context → SQL
# =============================================================================

import requests
import json

# Get one message with attachments to demonstrate
print("🔍 Getting a sample message with attachments...")
sample_msg = spark.sql("""
    SELECT workspace_id, space_id, conversation_id, message_id, attachments_count
    FROM genie_message_details_validation
    WHERE attachments_count > 0 AND statement_id IS NOT NULL
    LIMIT 1
""").collect()[0]

print(f"   Message ID: {sample_msg['message_id']}")
print(f"   Attachments: {sample_msg['attachments_count']}")

# Fetch from API
api_url = f"{workspace_url}/api/2.0/genie/spaces/{sample_msg['space_id']}/conversations/{sample_msg['conversation_id']}/messages/{sample_msg['message_id']}"
response = requests.get(api_url, headers=HEADERS, timeout=10)

if response.status_code == 200:
    msg_data = response.json()
    
    print(f"\n✅ API Response received\n")
    print("="*80)
    print("GOAL 1: CAPTURE COMPLETE CONTEXT AND SQL")
    print("="*80)
    
    # 1. Message Identifiers
    print(f"\n🏷️  IDENTIFIERS:")
    print(f"   space_id: {sample_msg['space_id']}")
    print(f"   conversation_id: {sample_msg['conversation_id']}")
    print(f"   message_id: {sample_msg['message_id']}")
    
    # 2. User Input
    print(f"\n📝 USER INPUT:")
    print(f"   Question: {msg_data.get('content', 'N/A')}")
    print(f"   Status: {msg_data.get('status', 'N/A')}")
    
    # 3. Context Sent to LLM (Attachments)
    attachments = msg_data.get('attachments', [])
    print(f"\n📎 CONTEXT SENT TO LLM ({len(attachments)} attachments):")
    
    query_sql_extracted = None
    query_description_extracted = None
    statement_id_extracted = None
    
    for idx, attachment in enumerate(attachments, 1):
        print(f"\n   Attachment {idx}:")
        print(f"      attachment_id: {attachment.get('attachment_id', 'N/A')}")
        
        # Query attachment
        if 'query' in attachment:
            query_obj = attachment['query']
            statement_id_extracted = query_obj.get('statement_id')
            query_sql_extracted = query_obj.get('query')
            query_description_extracted = query_obj.get('description')
            
            print(f"      type: QUERY")
            print(f"      statement_id: {statement_id_extracted}")
            print(f"      description: {query_description_extracted[:100]}..." if query_description_extracted else "      description: N/A")
            print(f"      SQL length: {len(query_sql_extracted)} characters" if query_sql_extracted else "      SQL: N/A")
            if query_sql_extracted:
                print(f"      SQL preview: {query_sql_extracted[:200]}...")
        
        # Text attachment
        elif 'text' in attachment:
            text_content = attachment['text'].get('content', '')
            print(f"      type: TEXT")
            print(f"      content: {text_content[:150]}..." if text_content else "      content: N/A")
        
        # Other attachment types
        else:
            print(f"      type: OTHER")
            print(f"      keys: {list(attachment.keys())}")
    
    # 4. SQL Generated by LLM
    print(f"\n⚡ SQL GENERATED BY LLM:")
    if query_sql_extracted:
        print(f"   ✅ SQL extracted successfully!")
        print(f"   Length: {len(query_sql_extracted)} characters")
        print(f"   Statement ID: {statement_id_extracted}")
        print(f"\n   Full SQL:")
        print(f"   {'-'*76}")
        print(f"   {query_sql_extracted}")
        print(f"   {'-'*76}")
    else:
        print(f"   ❌ No SQL found in attachments")
    
    print(f"\n\n💡 KEY INSIGHT:")
    print(f"   The SQL is stored in: attachments[i]['query']['query']")
    print(f"   NOT in: msg_data['query']['query']")
    print(f"\n   Cell 5.4 needs to be re-run with updated code to extract SQL properly.")
    
else:
    print(f"❌ API call failed: {response.status_code}")

🔍 Getting a sample message with attachments...
   Message ID: 01f0fd344afc14158579397bd3cacb76
   Attachments: 3

✅ API Response received

GOAL 1: CAPTURE COMPLETE CONTEXT AND SQL

🏷️  IDENTIFIERS:
   space_id: 01f0aaabfed41a778b2e5302795ce495
   conversation_id: 01f0fd34179e1edd8ded89bd6e85a671
   message_id: 01f0fd344afc14158579397bd3cacb76

📝 USER INPUT:
   Question: What is the total quantity needed for each part based on the sales forecast?
   Status: COMPLETED

📎 CONTEXT SENT TO LLM (3 attachments):

   Attachment 1:
      attachment_id: 01f0fd344d8418568b7c572987219549
      type: QUERY
      statement_id: 01f0fd34-4d8b-1472-8da0-4eed2fab7120
      description: You want to see the total quantity needed for each part based on the sales forecast for all vehicle ...
      SQL length: 409 characters
      SQL preview: SELECT pm.part_number, pm.part_name, SUM(sf.vehicle_quantity * bom.quantity_used) AS total_quantity_needed FROM `bk`.`supply_chain_sop`.`sales_forecasts` sf JOIN `bk`.

## Attachment Structure in Genie Messages

### Overview

Attachments provide **context to the LLM** when generating responses. Each message can have multiple attachments containing queries, text, charts, and other data.

---

### Attachment Types

#### **1. QUERY Attachments**
Contains SQL queries executed as part of the conversation.

**Fields:**
* `attachment_id` (STRING) - Unique identifier for the attachment
* `query.statement_id` (STRING) - Links to `system.query.history` for performance metrics
* `query.query` (STRING) - The actual SQL query text
* `query.description` (STRING) - Human-readable description of what the query does
* `query.query_result_metadata` (OBJECT) - Metadata about query results

**Use Cases:**
* Track which queries were executed for each message
* Join with `system.query.history` using `statement_id` for query performance
* Analyze SQL patterns generated by Genie
* Identify failed queries and error patterns

---

#### **2. TEXT Attachments**
Contains text content sent to the LLM as context.

**Fields:**
* `attachment_id` (STRING) - Unique identifier
* `text.content` (STRING) - Text content (instructions, clarifications, follow-ups)

**Use Cases:**
* Track conversation flow and context
* Analyze LLM prompts and instructions
* Understand what additional context was provided

---

#### **3. CHART Attachments**
Contains visualization metadata.

**Fields:**
* `attachment_id` (STRING) - Unique identifier
* `chart.type` (STRING) - Chart type (bar, line, pie, etc.)
* `chart.config` (OBJECT) - Chart configuration

**Use Cases:**
* Track visualization requests
* Analyze chart usage patterns

---

### Key Identifiers

#### **`attachment_id`**
* **Purpose:** Unique identifier for each attachment
* **Format:** UUID (e.g., `01f0fd3474941363af76060e4c19a9a8`)
* **Use:** Track what context was provided to the LLM for each message

#### **`statement_id`** (in QUERY attachments)
* **Purpose:** Links to `system.query.history` for query execution details
* **Format:** UUID (e.g., `01f0fd34-749b-1deb-93fa-e13ffe8b2765`)
* **Use:** Join with `system.query.history` to get:
  * Query execution time (`total_duration_ms`)
  * Rows read/written (`read_rows`, `written_rows`)
  * Execution status (`execution_status`)
  * Error messages (`error_message`)

---

### Example: Joining Attachments with Query History

```sql
-- Get message details with query performance metrics
SELECT 
  m.message_id,
  m.content as user_question,
  m.status as message_status,
  
  -- Attachment details (from API)
  a.attachment_id,
  a.query_sql,
  a.statement_id,
  
  -- Query performance (from system.query.history)
  qh.execution_status,
  qh.total_duration_ms,
  qh.read_rows,
  qh.read_bytes,
  qh.error_message
  
FROM genie_message_details m
LATERAL VIEW EXPLODE(attachments) AS a  -- Explode attachments array
LEFT JOIN system.query.history qh
  ON a.statement_id = qh.statement_id
  
WHERE m.attachments_count > 0
```

---

### Data Flow

```
User Question
     ↓
Genie generates SQL query
     ↓
Query executed (statement_id created)
     ↓
Query results attached to message (attachment_id created)
     ↓
Results + context sent to LLM
     ↓
LLM generates response
     ↓
Response shown to user
```

---

### Analytics Use Cases

1. **Query Performance Analysis**
   * Join attachments with `system.query.history` using `statement_id`
   * Identify slow queries, failed queries, expensive queries

2. **Context Tracking**
   * Use `attachment_id` to track what data was sent to LLM
   * Analyze how much context is needed for successful responses

3. **Conversation Flow**
   * Track sequence of attachments in a conversation
   * Understand how context builds up over multiple messages

4. **Error Analysis**
   * Identify messages with failed query attachments
   * Correlate query failures with message failures

---

### Next Steps

1. **Extract attachments to separate table:**
   ```python
   # Explode attachments array into rows
   attachments_df = message_details_df.select(
       'message_id',
       explode('attachments').alias('attachment')
   ).select(
       'message_id',
       'attachment.attachment_id',
       'attachment.query.statement_id',
       'attachment.query.query',
       'attachment.text.content'
   )
   ```

2. **Join with query history for performance metrics**

3. **Build attachment-level analytics dashboard**

In [0]:
-- =============================================================================
-- Query: Attachment Pattern Analysis
-- =============================================================================
-- Purpose: Analyze how attachment count correlates with message success
-- =============================================================================

SELECT 
  workspace_name,
  
  -- Attachment distribution
  attachments_count,
  COUNT(*) as message_count,
  
  -- Success metrics by attachment count
  COUNT(CASE WHEN status = 'COMPLETED' THEN 1 END) as completed_messages,
  COUNT(CASE WHEN status = 'FAILED' THEN 1 END) as failed_messages,
  ROUND(COUNT(CASE WHEN status = 'COMPLETED' THEN 1 END) * 100.0 / COUNT(*), 2) as completion_rate_pct,
  
  -- Query execution
  COUNT(CASE WHEN statement_id IS NOT NULL THEN 1 END) as messages_with_queries,
  ROUND(COUNT(CASE WHEN statement_id IS NOT NULL THEN 1 END) * 100.0 / COUNT(*), 2) as query_execution_rate_pct,
  
  -- Context level classification
  CASE 
    WHEN attachments_count = 0 THEN 'No context'
    WHEN attachments_count = 1 THEN 'Minimal context'
    WHEN attachments_count BETWEEN 2 AND 3 THEN 'Moderate context'
    ELSE 'Rich context'
  END as context_level
  
FROM genie_message_details_validation
WHERE api_fetch_status = 'SUCCESS'
GROUP BY workspace_name, attachments_count
ORDER BY attachments_count DESC;

workspace_name,attachments_count,message_count,completed_messages,failed_messages,completion_rate_pct,messages_with_queries,query_execution_rate_pct,context_level
DBW-BK-wx1,4,3,3,0,100.00,3,100.00,Rich context
DBW-BK-wx1,3,4,3,1,75.00,4,100.00,Moderate context
DBW-BK-wx1,2,6,5,1,83.33,4,66.67,Moderate context
DBW-BK-wx1,1,2,2,0,100.00,0,0.00,Minimal context
